In [1]:
import os
import random
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import warnings
from itertools import combinations as _combinations
from collections import Counter

from xgboost import XGBRegressor  

from IPython.display import display

import numpy as np
import pandas as pd

from datetime import datetime, timezone

from sklearn.model_selection import GroupKFold, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

SEED = 19537
EXTRA_SEEDS = [19537, 1584678, 17052356]  
SEEDS_TO_RUN = EXTRA_SEEDS

def seed_suffix(seed: int) -> str:
    return f"_seed{seed}"

def set_seeds(seed: int) -> None:
    global SEED, RNG, XGB_PARAMS
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    RNG = np.random.default_rng(SEED)
    XGB_PARAMS["random_state"] = SEED


os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(
    "ignore",
    message=".*Falling back to prediction using DMatrix due to mismatched devices.*",
    category=UserWarning,
)

In [2]:
# Paths and configuration

NOTEBOOK_SUBDIR = "notebook 3b"

ARTIFACTS = Path("artifacts")
IN_CLEAN = ARTIFACTS / "cleaned" / "notebook 2"
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"

OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR

for d in [OUT_REPORTS, OUT_META]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(SEED)

# Benchmark settings 
PRIMARY_TARGET = "auc"
N_DRUGS_TOP_BY_COVERAGE = 100      
MIN_MEASURED_CELLS_PER_DRUG = 120   # skip drugs that become too small after arm filtering
N_SPLITS_DESIRED = 10

# Preprocessing (leakage-safe: everything fit on train fold cell lines only)
USE_PCA = True
PCA_COMPONENTS = 200

# Models
RIDGE_ALPHA = 1.0

# Elastic Net (strong linear baseline with correlated features)
USE_ELASTICNET = True
EN_ALPHA = 0.05
EN_L1_RATIO = 0.2

# Forests (tree ensembles)
USE_EXTRATREES = True
ET_N_ESTIMATORS = 400
ET_MAX_DEPTH = None
ET_MIN_SAMPLES_LEAF = 2

USE_RANDOMFOREST = True  
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 2

# XGB
USE_XGB = True

USE_GPU_FOR_XGB = True     # set False to force CPU
XGB_DEVICE = "cuda"        # or "cuda:0"

XGB_PARAMS = dict(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,

    random_state=SEED,

    tree_method="hist",
    device=(XGB_DEVICE if USE_GPU_FOR_XGB else "cpu"),

    n_jobs=-1,
)

# Progress logging (stdout)
PRINT_EVERY_N_DRUGS = 5      # prints one progress line every N drugs per arm
PRINT_PER_FOLD = False       # True prints per-fold sizes (can be noisy)
PRINT_SKIPS = True           # True prints first few skip reasons per arm
MAX_SKIP_PRINTS = 5     
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_DRUGS = 5  # save state every N processed drugs per arm
def set_paths_for_seed(seed: int) -> dict:
    suf = seed_suffix(seed)
    paths = {
        "checkpoint_rows": OUT_REPORTS / f"prot_backbone_bench_perdrug__checkpoint_auc{suf}.csv",
        "checkpoint_state": OUT_META / f"prot_backbone_bench_state_auc{suf}.json",
        "detail": OUT_REPORTS / f"prot_backbone_bench_perdrug_auc{suf}.csv",
        "perdrug_agg": OUT_REPORTS / f"prot_backbone_bench_perdrug_aggregated_auc{suf}.csv",
        "metrics": OUT_REPORTS / f"prot_backbone_bench_metrics_auc{suf}.csv",
    }
    return paths

In [3]:
# Helpers

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def fingerprint(path: Path) -> dict:
    st = path.stat()
    return {
        "path": str(path.resolve()),
        "size_bytes": int(st.st_size),
        "mtime_utc": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
        "sha256": sha256_file(path),
    }

def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def read_json_if_exists(path: Path) -> Optional[dict]:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

BENCH_COLUMNS = [
    "seed",
    "arm", "compound_id", "fold", "model", "feature_set",
    "n_train", "n_test", "spearman", "r2",
]

def make_run_signature(selected_drugs: List[str], feature_combos: Dict[str, Tuple[str, ...]]) -> str:
    payload = {
        "PRIMARY_TARGET": PRIMARY_TARGET,
        "N_DRUGS_TOP_BY_COVERAGE": N_DRUGS_TOP_BY_COVERAGE,
        "MIN_MEASURED_CELLS_PER_DRUG": MIN_MEASURED_CELLS_PER_DRUG,
        "N_SPLITS_DESIRED": N_SPLITS_DESIRED,
        "USE_PCA": USE_PCA,
        "PCA_COMPONENTS": PCA_COMPONENTS,
        "RIDGE_ALPHA": RIDGE_ALPHA,
        "USE_ELASTICNET": USE_ELASTICNET,
        "EN_ALPHA": EN_ALPHA,
        "EN_L1_RATIO": EN_L1_RATIO,
        "USE_EXTRATREES": USE_EXTRATREES,
        "ET_N_ESTIMATORS": ET_N_ESTIMATORS,
        "ET_MAX_DEPTH": ET_MAX_DEPTH,
        "ET_MIN_SAMPLES_LEAF": ET_MIN_SAMPLES_LEAF,
        "USE_RANDOMFOREST": USE_RANDOMFOREST,
        "RF_N_ESTIMATORS": RF_N_ESTIMATORS,
        "RF_MAX_DEPTH": RF_MAX_DEPTH,
        "RF_MIN_SAMPLES_LEAF": RF_MIN_SAMPLES_LEAF,
        "USE_XGB": USE_XGB,
        "USE_GPU_FOR_XGB": USE_GPU_FOR_XGB,
        "XGB_DEVICE": XGB_DEVICE,
        "XGB_PARAMS": XGB_PARAMS,
        "FEATURE_COMBOS": {k: list(v) for k, v in feature_combos.items()},
        "selected_drugs": list(selected_drugs),
    }
    s = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha256(s).hexdigest()[:16]

def append_rows_csv(rows: List[dict], path: Path) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    # enforce stable column order
    for c in BENCH_COLUMNS:
        if c not in df.columns:
            df[c] = np.nan
    df = df[BENCH_COLUMNS]
    header = not path.exists()
    df.to_csv(path, mode="a", index=False, header=header)

def load_checkpoint(run_signature: str) -> Tuple[List[dict], Dict[str, set], int]:
    bench_rows: List[dict] = []
    processed_by_arm: Dict[str, set] = {}
    saved_len = 0

    if not RESUME_FROM_CHECKPOINT:
        return bench_rows, processed_by_arm, saved_len

    state = read_json_if_exists(CHECKPOINT_STATE_JSON)
    if state is None:
        return bench_rows, processed_by_arm, saved_len

    if state.get("run_signature") != run_signature:
        print("[checkpoint] Signature mismatch, ignoring existing checkpoint.")
        return bench_rows, processed_by_arm, saved_len

    if CHECKPOINT_ROWS_CSV.exists():
        df = pd.read_csv(CHECKPOINT_ROWS_CSV)
        bench_rows = df.to_dict(orient="records")
        saved_len = len(bench_rows)

    raw = state.get("processed_by_arm", {})
    processed_by_arm = {arm: set(drugs) for arm, drugs in raw.items()}

    print(f"[checkpoint] Resumed: rows={saved_len}, arms_with_progress={len(processed_by_arm)}")
    return bench_rows, processed_by_arm, saved_len

def save_checkpoint(run_signature: str, bench_rows: List[dict], processed_by_arm: Dict[str, set], saved_len_box: dict) -> None:
    new_rows = bench_rows[saved_len_box["saved_len"]:]
    append_rows_csv(new_rows, CHECKPOINT_ROWS_CSV)
    saved_len_box["saved_len"] = len(bench_rows)

    state = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "run_signature": run_signature,
        "saved_rows": int(saved_len_box["saved_len"]),
        "processed_by_arm": {arm: sorted(list(s)) for arm, s in processed_by_arm.items()},
    }
    write_json(state, CHECKPOINT_STATE_JSON)
    if new_rows:
        print(f"[checkpoint] Wrote +{len(new_rows)} rows (total={saved_len_box['saved_len']})")
    else:
        print(f"[checkpoint] State saved (total_rows={saved_len_box['saved_len']})")

def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)

def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df

def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    # rank transform with average ties
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])

def pick_group_column(cell_index: pd.DataFrame) -> str:
    candidates = ["lineage_1", "primary_disease", "lineage", "lineage_2"]
    for c in candidates:
        if c in cell_index.columns:
            return c
    return "depmap_id"

def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    """
    Returns list of (train_idx, test_idx) splits over 'cells' indices.
    Falls back to KFold if GroupKFold is not feasible.
    """
    groups = groups.reindex(cells)
    groups = groups.fillna("Unknown").astype(str)

    n_groups = groups.nunique()
    n_cells = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)

    if n_splits >= 2 and n_groups >= 2:
        splitter = GroupKFold(n_splits=n_splits)
        splits = list(splitter.split(X=np.zeros((n_cells, 1)), y=np.zeros(n_cells), groups=groups.values))
        return splits, f"GroupKFold(n_splits={n_splits})"
    # fallback
    n_splits = min(max(2, min(n_splits_desired, n_cells)), n_cells)
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    splits = list(splitter.split(np.zeros((n_cells, 1))))
    return splits, f"KFold(n_splits={n_splits}, shuffle=True, random_state={SEED})"

def has_any_observation(df: pd.DataFrame) -> pd.Series:
    if df.shape[1] == 0:
        return pd.Series(False, index=df.index)
    return df.notna().any(axis=1)

class FoldTransformer:
    """
    Imputer + scaler + optional PCA, fitted on train-fold cell lines only (leakage-safe),
    then used to transform any set of cell lines.
    """
    def __init__(self, use_pca: bool, n_components: int, random_state: int = 0):
        self.use_pca = bool(use_pca)
        self.n_components = int(n_components)
        self.random_state = int(random_state)

        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler(with_mean=True, with_std=True)
        self.pca = None

        self.keep_mask: Optional[np.ndarray] = None  # bool mask over input columns
        self.n_dropped_all_missing: int = 0

    def fit(self, X_train: np.ndarray) -> "FoldTransformer":
        X_train = np.asarray(X_train, dtype=float)

        if X_train.ndim != 2 or X_train.shape[1] == 0:
            self.keep_mask = np.zeros((0,), dtype=bool)
            self.n_dropped_all_missing = 0
            self.pca = None
            return self

        # keep columns with at least one finite observation in TRAIN
        keep = np.isfinite(X_train).any(axis=0)
        self.keep_mask = keep.astype(bool)
        self.n_dropped_all_missing = int((~self.keep_mask).sum())

        # if everything is missing in this fold, output empty embeddings for this modality
        if int(self.keep_mask.sum()) == 0:
            self.pca = None
            return self

        Xk = X_train[:, self.keep_mask]
        X_imp = self.imputer.fit_transform(Xk)
        X_std = self.scaler.fit_transform(X_imp)

        if self.use_pca:
            n = X_std.shape[0]
            d = X_std.shape[1]
            n_comp = min(self.n_components, max(1, n - 1), d)
            self.pca = PCA(n_components=n_comp, random_state=self.random_state)
            self.pca.fit(X_std)

        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)

        if self.keep_mask is None or int(self.keep_mask.sum()) == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)

        Xk = X[:, self.keep_mask]
        X_imp = self.imputer.transform(Xk)
        X_std = self.scaler.transform(X_imp)

        if self.pca is not None:
            X_std = self.pca.transform(X_std)

        return X_std.astype(np.float32, copy=False)

def try_make_xgb():
    if not USE_XGB:
        return None, "disabled"
    try:
        try:
            mdl = XGBRegressor(**XGB_PARAMS)
            name = f"xgboost.XGBRegressor(device={XGB_PARAMS.get('device')}, tree_method={XGB_PARAMS.get('tree_method')})"
            return mdl, name
        except TypeError:
            params = dict(XGB_PARAMS)
            params.pop("device", None)
            params["tree_method"] = "gpu_hist" if USE_GPU_FOR_XGB else "hist"
            if USE_GPU_FOR_XGB:
                params["predictor"] = "gpu_predictor"
            mdl = XGBRegressor(**params)
            name = f"xgboost.XGBRegressor(tree_method={params['tree_method']})"
            return mdl, name

    except Exception:
        try:
            return HistGradientBoostingRegressor(random_state=SEED), "sklearn.HistGradientBoostingRegressor (fallback)"
        except Exception:
            return None, "unavailable"
        
def make_models():
    models = []
    models.append(("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)))

    if USE_ELASTICNET:
        models.append((
            "elasticnet",
            ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1_RATIO, random_state=SEED, max_iter=10000)
        ))

    if USE_EXTRATREES:
        models.append((
            "extratrees",
            ExtraTreesRegressor(
                n_estimators=ET_N_ESTIMATORS,
                random_state=SEED,
                n_jobs=-1,
                max_depth=ET_MAX_DEPTH,
                min_samples_leaf=ET_MIN_SAMPLES_LEAF,
            )
        ))

    if USE_RANDOMFOREST:
        models.append((
            "randomforest",
            RandomForestRegressor(
                n_estimators=RF_N_ESTIMATORS,
                random_state=SEED,
                n_jobs=-1,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            )
        ))

    return models

In [4]:
# Load cleaned backbone (Notebook 2) and arm matrices (Notebook 1 Track 2)

cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

# Track 2 backbone matrices
rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

# proteomics per arm (Track 2)
prot_paths = {
    "prot_ms_ccle_gygi": IN_T2 / "prot_optional__prot_ms_ccle_gygi.parquet",
    "prot_rppa_ccle": IN_T2 / "prot_optional__prot_rppa_ccle.parquet",
    "prot_procan_depmapSanger": IN_T2 / "prot_optional__prot_procan_depmapSanger.parquet",
    "prot_combined_union": IN_T2 / "prot_optional__prot_combined_union.parquet",
}

proteomics_arms: Dict[str, pd.DataFrame] = {}
for arm, p in prot_paths.items():
    if p.exists():
        df = normalise_str_index(pd.read_parquet(p))
        proteomics_arms[arm] = df
    else:
        proteomics_arms[arm] = pd.DataFrame(index=pd.Index([], name="depmap_id"))

# Normalise cell_index depmap_id
if "depmap_id" not in cell_index.columns:
    raise ValueError("cell_index.parquet must include depmap_id column.")
cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()

group_col = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string")
    .fillna("Unknown")
    .astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print("Loaded core Track 2 cohort:", len(core_cells))
print("Group column for CV:", group_col)

# PRISM cohorts
prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()

prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][["depmap_id", "compound_id", "y"]].copy()
cells_prism_auc = set(prism_auc["depmap_id"].unique().tolist())

Loaded core Track 2 cohort: 1079
Group column for CV: lineage_1


In [5]:
# Coverage + overlap table per arm

def summarise_missingness(df: pd.DataFrame) -> dict:
    if df.shape[0] == 0 or df.shape[1] == 0:
        return dict(
            overall_missing_pct=np.nan,
            col_missing_q50=np.nan,
            col_missing_q90=np.nan,
            col_missing_q99=np.nan,
            row_missing_q50=np.nan,
            row_missing_q90=np.nan,
            row_missing_q99=np.nan,
        )
    col_miss = df.isna().mean(axis=0)
    row_miss = df.isna().mean(axis=1)
    return dict(
        overall_missing_pct=float(df.isna().mean().mean() * 100.0),
        col_missing_q50=float(col_miss.quantile(0.50)),
        col_missing_q90=float(col_miss.quantile(0.90)),
        col_missing_q99=float(col_miss.quantile(0.99)),
        row_missing_q50=float(row_miss.quantile(0.50)),
        row_missing_q90=float(row_miss.quantile(0.90)),
        row_missing_q99=float(row_miss.quantile(0.99)),
    )

def union_platform_availability_diag(union_df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Diagnostics for Arm D (namespaced union):
    - which platform blocks are present per cell line
    - approximate how much missingness is due to entire platform absence
    """
    if union_df.shape[0] == 0 or union_df.shape[1] == 0:
        return pd.DataFrame(), {}

    prefixes = ["ms__", "rppa__", "procan__"]
    blocks = {pref: [c for c in union_df.columns if str(c).startswith(pref)] for pref in prefixes}

    # block present if any non-missing values in that block
    present = pd.DataFrame(index=union_df.index)
    for pref, cols in blocks.items():
        if cols:
            present[pref[:-2]] = union_df[cols].notna().any(axis=1).astype(np.int8)
        else:
            present[pref[:-2]] = 0

    # pattern label per cell line
    def patt(row) -> str:
        inc = [k for k in ["ms", "rppa", "procan"] if int(row.get(k, 0)) == 1]
        return "+".join(inc) if inc else "none"

    patterns = present.apply(patt, axis=1).rename("pattern")
    pat_counts = patterns.value_counts().rename_axis("pattern").reset_index(name="n_cells")
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])

    # missingness contribution from full-block absence
    contrib = {}
    for pref, cols in blocks.items():
        if not cols:
            continue
        key = pref[:-2]  # ms, rppa, procan
        absent_mask = present[key].eq(0).to_numpy()
        n_absent = int(absent_mask.sum())
        n_cols = len(cols)
        # entries are fully missing when platform absent by construction
        missing_from_absence = n_absent * n_cols
        missing_total = int(union_df[cols].isna().sum().sum())
        contrib[f"{key}_frac_cells_present"] = float(present[key].mean())
        contrib[f"{key}_missing_absence_contrib"] = float(missing_from_absence / missing_total) if missing_total > 0 else np.nan

    return pat_counts, contrib

coverage_rows = []
union_patterns_df = None
union_platform_stats = {}

for arm, prot in proteomics_arms.items():
    prot = prot.copy()
    prot.index = prot.index.astype(str).str.strip()

    # only consider rows that are in the core cohort
    prot_core = prot.reindex(core_cells)

    # cell lines with any proteomics observation (arm-specific)
    has_prot = has_any_observation(prot_core)
    cells_with_prot = set(has_prot[has_prot].index.tolist())

    row = {
    "arm": arm,
    "n_cells_total_in_core": int(prot_core.shape[0]),
    "n_features": int(prot_core.shape[1]),
    "n_cells_with_any_prot": int(len(cells_with_prot)),
    "n_overlap_with_prism_auc_cells": int(len(cells_with_prot & cells_prism_auc)),
    }

    miss_stats = summarise_missingness(prot_core.loc[sorted(cells_with_prot)] if len(cells_with_prot) else prot_core)
    row.update(miss_stats)

    if arm == "prot_combined_union":
        union_patterns_df, union_platform_stats = union_platform_availability_diag(prot_core.loc[sorted(cells_with_prot)])
        row.update(union_platform_stats)

    coverage_rows.append(row)

prot_backbone_coverage = pd.DataFrame(coverage_rows).sort_values("n_overlap_with_prism_auc_cells", ascending=False)
coverage_path = OUT_REPORTS / "prot_backbone_coverage_auc.csv"
prot_backbone_coverage.to_csv(coverage_path, index=False)
print("Wrote:", coverage_path)

if union_patterns_df is not None and union_patterns_df.shape[0] > 0:
    union_patterns_path = OUT_REPORTS / "prot_combined_union__platform_availability_patterns_auc.csv"
    union_patterns_df.to_csv(union_patterns_path, index=False)
    print("Wrote:", union_patterns_path)

Wrote: artifacts/reports/notebook 3b/prot_backbone_coverage_auc.csv
Wrote: artifacts/reports/notebook 3b/prot_combined_union__platform_availability_patterns_auc.csv


In [6]:
# Quick benchmark: Ridge + XGB on backbone vs backbone+PROT per arm

# Select drugs by coverage in PRISM auc (not by performance)
drug_cov = prism_auc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
selected_drugs = drug_cov.head(N_DRUGS_TOP_BY_COVERAGE).index.tolist()
print(f"Selected top-{len(selected_drugs)} drugs by PRISM auc coverage.")

# Pull only the relevant pairs for the selected drugs to keep memory and grouping fast
prism_sel = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}

xgb_model, xgb_name = try_make_xgb()
print("Non-linear model:", xgb_name)

MODALITIES = ("rna", "cnv", "mut", "prot")

FEATURE_COMBOS: Dict[str, Tuple[str, ...]] = {}
for r in range(1, len(MODALITIES) + 1):
    for combo in _combinations(MODALITIES, r):
        FEATURE_COMBOS["+".join(combo)] = combo

def fit_fold_embeddings(
    eligible_cells: List[str],
    train_cells: List[str],
    arm: str,
    prot_df: pd.DataFrame,
) -> Tuple[Dict[str, np.ndarray], Dict[str, int]]:
    """
    Fit per-modality transformers on train_cells only, transform all eligible_cells.
    Returns:
      mats: dict with keys: rna, cnv, mut, prot (each is [n_cells, d])
      info: dims and dropped-all-missing counts
    """
    rna_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 0).fit(rna.loc[train_cells].to_numpy())
    cnv_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 1).fit(cnv.loc[train_cells].to_numpy())
    mut_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 2).fit(mut.loc[train_cells].to_numpy())

    Xr = rna_tr.transform(rna.loc[eligible_cells].to_numpy())
    Xc = cnv_tr.transform(cnv.loc[eligible_cells].to_numpy())
    Xm = mut_tr.transform(mut.loc[eligible_cells].to_numpy())

    Xp = np.zeros((len(eligible_cells), 0), dtype=np.float32)
    dropped_all_missing_prot = 0
    if prot_df is not None and prot_df.shape[1] > 0:
        prot_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 3).fit(
            prot_df.loc[train_cells].to_numpy()
        )
        Xp = prot_tr.transform(prot_df.loc[eligible_cells].to_numpy())
        dropped_all_missing_prot = int(getattr(prot_tr, "n_dropped_all_missing", 0))

    mats = {"rna": Xr, "cnv": Xc, "mut": Xm, "prot": Xp}

    info = {
        "d_rna": int(Xr.shape[1]),
        "d_cnv": int(Xc.shape[1]),
        "d_mut": int(Xm.shape[1]),
        "d_prot": int(Xp.shape[1]),
        "dropped_all_missing_rna": int(getattr(rna_tr, "n_dropped_all_missing", 0)),
        "dropped_all_missing_cnv": int(getattr(cnv_tr, "n_dropped_all_missing", 0)),
        "dropped_all_missing_mut": int(getattr(mut_tr, "n_dropped_all_missing", 0)),
        "dropped_all_missing_prot": int(dropped_all_missing_prot),
    }
    return mats, info

for run_seed in SEEDS_TO_RUN:
    set_seeds(run_seed)
    p = set_paths_for_seed(run_seed)
    suf = seed_suffix(run_seed)
    # point the existing checkpoint helpers at the seed-specific files
    CHECKPOINT_ROWS_CSV = p["checkpoint_rows"]
    CHECKPOINT_STATE_JSON = p["checkpoint_state"]

    # skip: if this seed already finished, do nothing
    if p["detail"].exists() and p["perdrug_agg"].exists() and p["metrics"].exists():
        print(f"[seed={run_seed}] Outputs already exist, skipping.")
        continue

    print(f"\n=== Running auc seed {run_seed} ===")

    # Checkpoint load (resume)
    RUN_SIGNATURE = make_run_signature(selected_drugs=selected_drugs, feature_combos=FEATURE_COMBOS)
    bench_rows, processed_by_arm, _saved_len = load_checkpoint(RUN_SIGNATURE)
    _ckpt = {"saved_len": int(_saved_len)}

    for r in bench_rows:
        if ("seed" not in r) or pd.isna(r.get("seed")):
            r["seed"] = int(SEED)

    for arm, prot in proteomics_arms.items():
        if prot.shape[0] == 0 or prot.shape[1] == 0:
            print(f"[WARN] Skipping benchmark for {arm} (no proteomics loaded).")
            continue
        
        # Resume support per arm
        processed_drugs = processed_by_arm.get(arm, set())
        processed_by_arm[arm] = processed_drugs
        processed_this_run = 0

        prot = prot.copy()
        prot.index = prot.index.astype(str).str.strip()
        prot_core = prot.reindex(core_cells)

        # Define eligible cells for this arm as those with any proteomics values (so base vs base+prot is comparable)
        has_prot = has_any_observation(prot_core)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())

        if len(eligible_cells) < 200:
            print(f"[WARN] {arm}: too few eligible cells ({len(eligible_cells)}), skipping.")
            continue

        # Precompute CV splits once per arm (over eligible cells)
        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED)
        print(f"{arm}: CV={split_name}, eligible_cells={len(eligible_cells)}")

        # Build a fold map: cell_id -> fold index (test fold)
        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        # Pre-fit fold-specific embeddings (transformers are fitted without labels, so re-usable across drugs)
        fold_cache = {}
        for fold_i, (train_idx, test_idx) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]
            # Restrict prot_core to eligible cells, and align
            prot_elig = prot_core.loc[eligible_cells]
            mats, info = fit_fold_embeddings(
                eligible_cells=eligible_cells,
                train_cells=train_cells,
                arm=arm,
                prot_df=prot_elig,
            )

            cell_to_row = {cid: i for i, cid in enumerate(eligible_cells)}

            fold_cache[fold_i] = {
                "mats": mats,
                "cell_to_row": cell_to_row,
                "info": info,
                "train_cells_set": set(train_cells),
            }

        n_selected = len(selected_drugs)
        n_seen = 0
        n_evaluated = 0
        n_skipped_missing_pairs = 0
        n_skipped_low_cells = 0
        n_skipped_no_valid_folds = 0
        skip_prints_used = 0

        # Evaluate per drug
        for drug_i, drug in enumerate(selected_drugs, start=1):
            if drug in processed_drugs:
                continue
            n_seen += 1

            pairs = drug_to_pairs.get(drug)
            if pairs is None or pairs.shape[0] == 0:
                n_skipped_missing_pairs += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: no PRISM pairs found, skipping")
                    skip_prints_used += 1

                processed_drugs.add(drug)
                processed_this_run += 1
                if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                    save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)
                continue 

            # Restrict to eligible cells for this arm
            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            n_cells_raw = int(df["depmap_id"].nunique())

            if n_cells_raw < MIN_MEASURED_CELLS_PER_DRUG:
                n_skipped_low_cells += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(
                        f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: "
                        f"only {n_cells_raw} cells after arm filter (<{MIN_MEASURED_CELLS_PER_DRUG}), skipping"
                    )
                    skip_prints_used += 1
                processed_drugs.add(drug)
                processed_this_run += 1
                if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                    save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)
                continue
                

            # Ensure one measurement per cell line
            df = df.groupby("depmap_id", as_index=False)["y"].mean()

            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            # Determine fold id for each sample (based on depmap_id)
            fold_ids = np.array([fold_map.get(cid, -1) for cid in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_MEASURED_CELLS_PER_DRUG:
                n_skipped_low_cells += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(
                        f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: "
                        f"{len(cell_ids)} cells after fold-map filter (<{MIN_MEASURED_CELLS_PER_DRUG}), skipping"
                    )
                    skip_prints_used += 1

                processed_drugs.add(drug)
                processed_this_run += 1
                if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                    save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)
                continue

            n_evaluated += 1

            if (drug_i % PRINT_EVERY_N_DRUGS == 0) or (drug_i == 1) or (drug_i == n_selected):
                print(f"[{arm}] ({drug_i}/{n_selected}) Evaluating drug={drug} with {len(cell_ids)} cells")

            any_fold_ran = False

            for fold_i in sorted(fold_cache.keys()):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 10 or n_train < 30:
                    continue

                any_fold_ran = True

                if PRINT_PER_FOLD:
                    print(f"    fold={fold_i}: n_train={n_train}, n_test={n_test}")

                cache = fold_cache[fold_i]
                cell_to_row = cache["cell_to_row"]
                mats = cache["mats"]

                idx_all = np.array([cell_to_row[cid] for cid in cell_ids], dtype=int)
                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]

                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                bench_rows.append({
                    "seed": int(SEED),
                    "arm": arm,
                    "compound_id": drug,
                    "fold": int(fold_i),
                    "model": "null_mean",
                    "feature_set": "none",
                    "n_train": int(len(y_train)),
                    "n_test": int(len(y_test)),
                    "spearman": 0.0,
                    "r2": float(r2_score(y_test, np.full_like(y_test, float(y_train.mean())))),
                })

                def build_X(combo_keys: Tuple[str, ...]) -> np.ndarray:
                    # Require every requested modality to be present, otherwise skip this combo
                    for k in combo_keys:
                        if mats[k].shape[1] == 0:
                            return np.zeros((len(eligible_cells), 0), dtype=np.float32)
                    return np.concatenate([mats[k] for k in combo_keys], axis=1)

                # Build feature matrices once per fold
                X_by_combo = {name: build_X(keys) for name, keys in FEATURE_COMBOS.items()}

                # Linear and forest models
                for model_name, model in make_models():
                    for feat_name, Xmat_all in X_by_combo.items():
                        if Xmat_all.shape[1] == 0:
                            continue

                        X_train = Xmat_all[idx_train]
                        X_test = Xmat_all[idx_test]

                        model.fit(X_train, y_train)
                        pred = model.predict(X_test)

                        bench_rows.append({
                            "seed": int(SEED),
                            "arm": arm,
                            "compound_id": drug,
                            "fold": int(fold_i),
                            "model": model_name,
                            "feature_set": feat_name,
                            "n_train": int(len(y_train)),
                            "n_test": int(len(y_test)),
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })

                # XGB (recreated per fit)
                if xgb_model is not None:
                    for feat_name, Xmat_all in X_by_combo.items():
                        if Xmat_all.shape[1] == 0:
                            continue

                        X_train = Xmat_all[idx_train]
                        X_test = Xmat_all[idx_test]

                        mdl, _ = try_make_xgb()
                        if mdl is None:
                            break

                        mdl.fit(X_train, y_train)
                        global _XGB_DEVICE_PRINTED
                        if not _XGB_DEVICE_PRINTED:
                            cfg = mdl.get_booster().save_config()
                            i = cfg.find('"device"')
                            print("XGB config snippet:", cfg[i:i+120] if i != -1 else "device not found in config")
                            _XGB_DEVICE_PRINTED = True

                        pred = mdl.predict(X_test)

                        bench_rows.append({
                            "seed": int(SEED),
                            "arm": arm,
                            "compound_id": drug,
                            "fold": int(fold_i),
                            "model": "xgb_quick",
                            "feature_set": feat_name,
                            "n_train": int(len(y_train)),
                            "n_test": int(len(y_test)),
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })


            if not any_fold_ran:
                n_skipped_no_valid_folds += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(
                        f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: "
                        "no folds met min sizes (test>=10 and train>=30), skipping"
                    )
                    skip_prints_used += 1

            processed_drugs.add(drug)
            processed_this_run += 1
            if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)

        # End-of-arm summary
        print(
            f"[{arm}] Done: seen={n_seen}, evaluated={n_evaluated}, "
            f"skipped_no_pairs={n_skipped_missing_pairs}, skipped_low_cells={n_skipped_low_cells}, "
            f"skipped_no_valid_folds={n_skipped_no_valid_folds}"
        )
        save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)


    bench_detail = pd.DataFrame(bench_rows)
    detail_path = p["detail"]
    bench_detail.to_csv(detail_path, index=False)
    print("Wrote:", detail_path, bench_detail.shape)

    # Summarise: per arm, model, feature_set
    if bench_detail.shape[0] == 0:
        raise RuntimeError("Benchmark produced no rows. Consider lowering MIN_MEASURED_CELLS_PER_DRUG or N_DRUGS_TOP_BY_COVERAGE.")

    # Per-drug mean across folds
    drug_means = (
        bench_detail
        .groupby(["seed", "arm", "model", "feature_set", "compound_id"], as_index=False)
        .agg(
            spearman=("spearman", "mean"),
            r2=("r2", "mean"),
            n_folds=("fold", "nunique"),
            n_test_total=("n_test", "sum"),
        )
    )

    bench_summary = (
        drug_means
        .groupby(["seed", "arm", "model", "feature_set"], as_index=False)
        .agg(
            n_drugs=("compound_id", "nunique"),
            mean_spearman=("spearman", "mean"),
            median_spearman=("spearman", "median"),
            mean_r2=("r2", "mean"),
            median_r2=("r2", "median"),
            mean_folds=("n_folds", "mean"),
        )
    )

    def add_uplift(df: pd.DataFrame) -> pd.DataFrame:
        base = df[df["feature_set"] == "rna+cnv+mut"].set_index(["seed", "arm", "model"])
        full = df[df["feature_set"] == "rna+cnv+mut+prot"].set_index(["seed", "arm", "model"])
        n_base = base.shape[0]
        n_full = full.shape[0]
        if n_base == 0:
            print("[WARN] add_uplift: no rows found for feature_set='rna+cnv+mut' — delta columns will be NaN.")
        if n_full == 0:
            print("[WARN] add_uplift: no rows found for feature_set='rna+cnv+mut+prot' — delta columns will be NaN.")
        merged = full.join(base, lsuffix="_full", rsuffix="_base", how="left")
        out = merged.reset_index()
        n_nan_delta = out["mean_spearman_full"].isna().sum() + out["mean_spearman_base"].isna().sum()
        if n_nan_delta > 0:
            print(f"[WARN] add_uplift: {n_nan_delta} NaN values in uplift join — some arms may lack a baseline or full feature set.")
        out["delta_mean_spearman"] = out["mean_spearman_full"] - out["mean_spearman_base"]
        out["delta_mean_r2"] = out["mean_r2_full"] - out["mean_r2_base"]
        return out

    uplift = add_uplift(bench_summary)

    metrics_path = p["metrics"]
    uplift.to_csv(metrics_path, index=False)
    print("Wrote:", metrics_path)

    coverage_df = prot_backbone_coverage.set_index("arm")

    uplift_ridge = uplift[uplift["model"] == "ridge"].copy()
    if uplift_ridge.shape[0] > 0:
        uplift_ridge["n_overlap_with_prism_auc_cells"] = uplift_ridge["arm"].map(
            lambda a: int(coverage_df.loc[a, "n_overlap_with_prism_auc_cells"]) if a in coverage_df.index else 0
        )
        uplift_ridge["overall_missing_pct"] = uplift_ridge["arm"].map(
            lambda a: float(coverage_df.loc[a, "overall_missing_pct"]) if a in coverage_df.index else np.nan
        )

        # Score: uplift + small bonus for overlap, small penalty for missingness
        uplift_ridge["score"] = (
            uplift_ridge["delta_mean_spearman"].fillna(-1e9)
            + 0.0002 * uplift_ridge["n_overlap_with_prism_auc_cells"].astype(float)
            - 0.001 * uplift_ridge["overall_missing_pct"].fillna(100.0)
        )
        uplift_ridge = uplift_ridge.sort_values("score", ascending=False)
        suggested = uplift_ridge["arm"].head(2).tolist()
    else:
        suggested = []

    drug_means_rank = drug_means.drop_duplicates(
        subset=["seed", "arm", "model", "feature_set", "compound_id", "spearman", "r2", "n_folds", "n_test_total"]
    ).copy()

    per_drug_agg_path = p["perdrug_agg"]
    drug_means_rank.to_csv(per_drug_agg_path, index=False)
    print("Wrote:", per_drug_agg_path)

    # Overall top/bottom 10 across all arms/models/feature sets
    overall_top10 = (
        drug_means_rank
        .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[False, False, False, False])
        .head(10)
        .reset_index(drop=True)
    )

    overall_bottom10 = (
        drug_means_rank
        .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[True, True, False, False])
        .head(10)
        .reset_index(drop=True)
    )

    overall_top10_path = OUT_REPORTS / f"prot_backbone_bench_top10_overall_auc{suf}.csv"
    overall_bottom10_path = OUT_REPORTS / f"prot_backbone_bench_bottom10_overall_auc{suf}.csv"
    overall_top10.to_csv(overall_top10_path, index=False)
    overall_bottom10.to_csv(overall_bottom10_path, index=False)
    print("Wrote:", overall_top10_path)
    print("Wrote:", overall_bottom10_path)

    # top/bottom within each group 
    def _top_bottom_by_group(g: pd.DataFrame, n: int = 10) -> pd.DataFrame:
        top = (
            g.sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[False, False, False, False])
            .head(n)
            .copy()
        )
        top["rank_block"] = "top"

        bottom = (
            g.sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[True, True, False, False])
            .head(n)
            .copy()
        )
        bottom["rank_block"] = "bottom"

        return pd.concat([top, bottom], ignore_index=True)

    def _top_bottom_by_group_loop(df: pd.DataFrame, group_cols: List[str], n: int = 10) -> pd.DataFrame:
        parts = []
        for _, g in df.groupby(group_cols, sort=False):
            parts.append(_top_bottom_by_group(g, n=n))
        if not parts:
            return pd.DataFrame(columns=list(df.columns) + ["rank_block"])
        return pd.concat(parts, ignore_index=True)

    # Top/bottom 10 within each (arm, model, feature_set)
    per_group_top_bottom = _top_bottom_by_group_loop(
        drug_means_rank,
        group_cols=["arm", "model", "feature_set"],
        n=10,
    )

    per_group_top_bottom_path = OUT_REPORTS / f"prot_backbone_bench_top_bottom10_by_group_auc{suf}.csv"
    per_group_top_bottom.to_csv(per_group_top_bottom_path, index=False)
    print("Wrote:", per_group_top_bottom_path)

Selected top-100 drugs by PRISM auc coverage.
Non-linear model: xgboost.XGBRegressor(device=cuda, tree_method=hist)
[seed=19537] Outputs already exist, skipping.
[seed=1584678] Outputs already exist, skipping.
[seed=17052356] Outputs already exist, skipping.


In [7]:
TARGET = PRIMARY_TARGET 
ALL_SEEDS = list(EXTRA_SEEDS)

def parse_seed_from_name(path: Path) -> Optional[int]:
    s = path.stem
    if "_seed" in s:
        try:
            return int(s.split("_seed")[-1])
        except Exception:
            return None
    return None

def load_perdrug_agg_for_seeds(out_dir: Path, seeds: List[int]) -> pd.DataFrame:
    """
    Loads per-drug aggregated outputs for multiple seeds into one DataFrame.
    It looks for:
      - per-seed files: set_paths_for_seed(seed)["perdrug_agg"]
      - plus the legacy base file without suffix (optional)
    """
    files = []

    for s in seeds:
        fp = set_paths_for_seed(s)["perdrug_agg"]
        if fp.exists():
            files.append(fp)
        else:
            print("Missing:", fp)

    legacy = out_dir / f"prot_backbone_bench_perdrug_aggregated_{TARGET}.csv"
    if legacy.exists():
        files.append(legacy)

    # De-dup
    files = list(dict.fromkeys(files))

    if not files:
        raise FileNotFoundError("No per-drug aggregated files found for the requested seeds.")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)

        if "seed" not in df.columns or df["seed"].isna().all():
            inferred = parse_seed_from_name(fp)
            df["seed"] = inferred

        df["seed"] = pd.to_numeric(df["seed"], errors="coerce")
        df = df.dropna(subset=["seed"]).copy()
        df["seed"] = df["seed"].astype(int)

        # Keep only requested seeds
        df = df[df["seed"].isin(seeds)].copy()

        # Numeric clean
        for c in ["spearman", "r2", "n_folds", "n_test_total"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True).drop_duplicates()
    return merged

merged = load_perdrug_agg_for_seeds(OUT_REPORTS, ALL_SEEDS)

merged_path = OUT_REPORTS / f"prot_backbone_bench_perdrug_aggregated_{TARGET}__merged_{len(ALL_SEEDS)}seeds.csv"
merged.to_csv(merged_path, index=False)
print("Wrote:", merged_path, merged.shape)

# Seed performance overall
seed_overall = (
    merged
    .groupby("seed", as_index=False)
    .agg(
        n_rows=("compound_id", "size"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        std_spearman=("spearman", "std"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
    )
    .sort_values(["mean_spearman", "median_spearman"], ascending=[False, False])
)

seed_overall_path = OUT_REPORTS / f"seed_overall_summary_{TARGET}.csv"
seed_overall.to_csv(seed_overall_path, index=False)
print("Wrote:", seed_overall_path)

best_seed = int(seed_overall.iloc[0]["seed"]) if seed_overall.shape[0] else None
worst_seed = int(seed_overall.iloc[-1]["seed"]) if seed_overall.shape[0] else None
print(f"Best seed overall: {best_seed}")
print(f"Worst seed overall: {worst_seed}")

display(seed_overall)

# Best and worst combo per seed
seed_combo = (
    merged
    .groupby(["seed", "arm", "model", "feature_set"], as_index=False)
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        std_spearman=("spearman", "std"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
    )
)

best_combo_per_seed = (
    seed_combo
    .sort_values(["seed", "mean_spearman", "mean_r2", "n_drugs"], ascending=[True, False, False, False])
    .groupby("seed", as_index=False)
    .head(1)
    .reset_index(drop=True)
    .assign(rank_block="best")
)

worst_combo_per_seed = (
    seed_combo
    .sort_values(["seed", "mean_spearman", "mean_r2", "n_drugs"], ascending=[True, True, True, False])
    .groupby("seed", as_index=False)
    .head(1)
    .reset_index(drop=True)
    .assign(rank_block="worst")
)

seed_best_worst_combo = (
    pd.concat([best_combo_per_seed, worst_combo_per_seed], ignore_index=True)
    .sort_values(["seed", "rank_block"])
)

seed_best_worst_combo_path = OUT_REPORTS / f"seed_best_worst_combo_{TARGET}.csv"
seed_best_worst_combo.to_csv(seed_best_worst_combo_path, index=False)
print("Wrote:", seed_best_worst_combo_path)

display(seed_best_worst_combo)

# Best and worst drugs across seeds
best_per_seed_drug = (
    merged
    .sort_values(
        ["seed", "compound_id", "spearman", "r2", "n_folds", "n_test_total"],
        ascending=[True, True, False, False, False, False]
    )
    .groupby(["seed", "compound_id"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

drug_across_seeds = (
    best_per_seed_drug
    .groupby("compound_id", as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        mean_best_spearman=("spearman", "mean"),
        median_best_spearman=("spearman", "median"),
        std_best_spearman=("spearman", "std"),
        mean_best_r2=("r2", "mean"),
    )
    .sort_values(["mean_best_spearman", "median_best_spearman"], ascending=[False, False])
)

def mode_combo(sub: pd.DataFrame) -> pd.Series:
    combos = list(zip(sub["arm"], sub["model"], sub["feature_set"]))
    m = Counter(combos).most_common(1)[0][0] if combos else ("", "", "")
    return pd.Series({"mode_arm": m[0], "mode_model": m[1], "mode_feature_set": m[2]})

combo_mode = best_per_seed_drug.groupby("compound_id").apply(mode_combo, include_groups=False).reset_index()
drug_across_seeds = drug_across_seeds.merge(combo_mode, on="compound_id", how="left")

drug_across_seeds_path = OUT_REPORTS / f"drug_bestcombo_across_seeds_{TARGET}.csv"
drug_across_seeds.to_csv(drug_across_seeds_path, index=False)
print("Wrote:", drug_across_seeds_path)

top_drugs = drug_across_seeds.head(15).reset_index(drop=True)
bottom_drugs = (
    drug_across_seeds
    .sort_values(["mean_best_spearman", "median_best_spearman"], ascending=[True, True])
    .head(15)
    .reset_index(drop=True)
)

top_drugs_path = OUT_REPORTS / f"top15_drugs_across_seeds_{TARGET}.csv"
bottom_drugs_path = OUT_REPORTS / f"bottom15_drugs_across_seeds_{TARGET}.csv"
top_drugs.to_csv(top_drugs_path, index=False)
bottom_drugs.to_csv(bottom_drugs_path, index=False)
print("Wrote:", top_drugs_path)
print("Wrote:", bottom_drugs_path)

print("\nTop drugs across seeds (mean of per-seed best combos):")
display(top_drugs)

print("\nBottom drugs across seeds (mean of per-seed best combos):")
display(bottom_drugs)

# Single best and worst observed rows overall
best_rows = (
    merged
    .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[False, False, False, False])
    .head(20)
    .reset_index(drop=True)
)

worst_rows = (
    merged
    .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[True, True, False, False])
    .head(20)
    .reset_index(drop=True)
)

best_rows_path = OUT_REPORTS / f"best20_rows_overall_{TARGET}.csv"
worst_rows_path = OUT_REPORTS / f"worst20_rows_overall_{TARGET}.csv"
best_rows.to_csv(best_rows_path, index=False)
worst_rows.to_csv(worst_rows_path, index=False)
print("Wrote:", best_rows_path)
print("Wrote:", worst_rows_path)

print("\nBest observed rows (seed + drug + combo):")
display(best_rows.head(10))

print("\nWorst observed rows (seed + drug + combo):")
display(worst_rows.head(10))


Wrote: artifacts/reports/notebook 3b/prot_backbone_bench_perdrug_aggregated_auc__merged_3seeds.csv (90000, 9)
Wrote: artifacts/reports/notebook 3b/seed_overall_summary_auc.csv
Best seed overall: 19537
Worst seed overall: 1584678


,seed,n_rows,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,median_r2
0,19537,30000,100,0.117508,0.112409,0.114512,-0.759788,-0.083176
2,17052356,30000,100,0.116796,0.110953,0.112494,-1.051928,-0.114290
1,1584678,30000,100,0.116736,0.110400,0.112294,-1.051962,-0.114434


Wrote: artifacts/reports/notebook 3b/seed_best_worst_combo_auc.csv


,seed,arm,model,feature_set,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,median_r2,rank_block
0,19537,prot_combined_union,elasticnet,rna,100,0.198903,0.213839,0.126132,-0.078213,-0.072513,best
3,19537,prot_procan_depmapSanger,randomforest,mut,100,-0.032891,-0.029625,0.069640,-0.099593,-0.087151,worst
1,1584678,prot_combined_union,elasticnet,rna,100,0.196414,0.209020,0.111051,-0.087193,-0.087166,best
4,1584678,prot_rppa_ccle,ridge,mut,100,-0.050409,-0.055333,0.063312,-9.860270,-6.537403,worst
2,17052356,prot_combined_union,elasticnet,rna,100,0.196414,0.209020,0.111051,-0.087193,-0.087166,best
5,17052356,prot_rppa_ccle,ridge,mut,100,-0.050409,-0.055333,0.063312,-9.860270,-6.537403,worst


Wrote: artifacts/reports/notebook 3b/drug_bestcombo_across_seeds_auc.csv
Wrote: artifacts/reports/notebook 3b/top15_drugs_across_seeds_auc.csv
Wrote: artifacts/reports/notebook 3b/bottom15_drugs_across_seeds_auc.csv

Top drugs across seeds (mean of per-seed best combos):


...

,compound_id,n_seeds,mean_best_spearman,median_best_spearman,std_best_spearman,mean_best_r2,mode_arm,mode_model,mode_feature_set
0,PI3K-IN-2 (BRD:BRD-K62374002-001-01-5),3,0.155572,0.156556,0.002824,-0.323804,prot_rppa_ccle,ridge,prot
1,VANOXERINE (BRD:BRD-K32501161-300-06-2),3,0.156196,0.166769,0.018312,-4.285128,prot_rppa_ccle,elasticnet,rna
2,TEROXIRONE (BRD:BRD-A39935389-001-05-9),3,0.161291,0.150568,0.018786,-0.290723,prot_combined_union,elasticnet,cnv+prot
3,LDN-212854 (BRD:BRD-K59831625-001-01-6),3,0.171350,0.151870,0.033741,-9.383089,prot_ms_ccle_gygi,ridge,cnv
4,FLUBENDAZOLE (BRD:BRD-K86003836-001-10-5),3,0.181561,0.188788,0.033159,-0.323340,prot_ms_ccle_gygi,xgb_quick,mut+prot
5,NH125 (BRD:BRD-K31086665-005-04-6),3,0.186702,0.189295,0.005020,-0.041689,prot_ms_ccle_gygi,randomforest,rna+prot
6,PP-121 (BRD:BRD-K81801188-001-02-8),3,0.189275,0.174801,0.044068,-0.076944,prot_ms_ccle_gygi,xgb_quick,rna+mut
7,RO-106-9920 (BRD:BRD-A53134341-001-03-2),3,0.190277,0.192036,0.003046,-0.254911,prot_ms_ccle_gygi,ridge,rna+cnv+mut
8,BIX-01294 (BRD:BRD-K26818574-305-04-3),3,0.197679,0.197768,0.003153,-0.703834,prot_ms_ccle_gygi,xgb_quick,rna+mut
9,PENFLURIDOL (BRD:BRD-K15409150-001-05-8),3,0.201503,0.205328,0.006625,-0.002429,prot_rppa_ccle,elasticnet,prot


Wrote: artifacts/reports/notebook 3b/best20_rows_overall_auc.csv
Wrote: artifacts/reports/notebook 3b/worst20_rows_overall_auc.csv

Best observed rows (seed + drug + combo):


,arm,model,feature_set,compound_id,spearman,r2,n_folds,n_test_total,seed
0,prot_combined_union,extratrees,rna+mut,SEPANTRONIUM BROMIDE (BRD:BRD-K76703230-004-04-1),0.495039,0.081366,5,364,19537
1,prot_procan_depmapSanger,elasticnet,rna+mut,SEPANTRONIUM BROMIDE (BRD:BRD-K76703230-004-04-1),0.492953,0.118957,9,269,1584678
2,prot_procan_depmapSanger,elasticnet,rna+mut,SEPANTRONIUM BROMIDE (BRD:BRD-K76703230-004-04-1),0.492953,0.118957,9,269,17052356
3,prot_procan_depmapSanger,extratrees,rna+prot,VERUBULIN (BRD:BRD-K42673188-001-01-1),0.488666,0.005119,9,265,1584678
4,prot_procan_depmapSanger,extratrees,rna+cnv+prot,VERUBULIN (BRD:BRD-K42673188-001-01-1),0.486254,0.042525,9,265,17052356
5,prot_combined_union,extratrees,rna+cnv,VELBAN (BRD:BRD-K06519765-065-01-6),0.483626,0.050509,9,362,17052356
6,prot_combined_union,extratrees,rna+prot,SEPANTRONIUM BROMIDE (BRD:BRD-K76703230-004-04-1),0.483270,0.082205,5,364,19537
7,prot_combined_union,extratrees,rna+cnv,VELBAN (BRD:BRD-K06519765-065-01-6),0.479453,0.047426,9,362,1584678
8,prot_combined_union,extratrees,rna+mut+prot,SEPANTRONIUM BROMIDE (BRD:BRD-K76703230-004-04-1),0.476863,0.073161,5,364,19537
9,prot_procan_depmapSanger,extratrees,rna+prot,SEPANTRONIUM BROMIDE (BRD:BRD-K76703230-004-04-1),0.474315,0.078611,5,269,19537



Worst observed rows (seed + drug + combo):


,arm,model,feature_set,compound_id,spearman,r2,n_folds,n_test_total,seed
0,prot_procan_depmapSanger,xgb_quick,cnv,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.317557,-0.610544,9,263,17052356
1,prot_procan_depmapSanger,extratrees,cnv,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.271938,-0.361523,9,263,1584678
2,prot_procan_depmapSanger,elasticnet,cnv+mut,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.256779,-0.710239,9,263,1584678
3,prot_procan_depmapSanger,elasticnet,cnv+mut,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.256779,-0.710239,9,263,17052356
4,prot_combined_union,elasticnet,cnv+mut,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.252024,-0.465155,5,359,19537
5,prot_combined_union,extratrees,rna+cnv+prot,WP1066 (BRD:BRD-K05445342-001-05-3),-0.251938,-0.154516,5,364,19537
6,prot_procan_depmapSanger,xgb_quick,cnv,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.251223,-0.550969,9,263,1584678
7,prot_procan_depmapSanger,randomforest,cnv,BIX-01294 (BRD:BRD-K26818574-305-04-3),-0.251205,-0.357518,9,263,1584678
8,prot_combined_union,extratrees,rna+cnv+mut+prot,WP1066 (BRD:BRD-K05445342-001-05-3),-0.246325,-0.133659,5,364,19537
9,prot_ms_ccle_gygi,ridge,mut,EPOTHILONE-A (BRD:BRD-K71823332-001-03-7),-0.237149,-3.728866,5,201,19537


# Overview and the Relationship to the LFC Analysis

Notebook 3b reproduces the full experimental structure of Notebook 3a across the same four proteomics arms, five model families, fifteen feature combinations, one hundred drugs, and three random seeds, with one fundamental modification: the drug response readout is now area under the dose-response curve (AUC) rather than log-fold-change (LFC) at a single fixed dose. The significance of this change lies in the nature of AUC itself and in why it constitutes a more tractable prediction target.

When a cell line is exposed to a compound across a range of concentrations and viability is measured at each point, the resulting dose-response curve captures the full sensitivity profile of that cell line to the compound. AUC summarises this curve into a single scalar by integrating the area beneath it. As a result, it reflects not merely whether a cell line is sensitive at one concentration, but how strongly and consistently it is sensitive across the full tested dose range. Because it aggregates information across multiple concentration points, AUC is substantially less noisy than single-dose LFC. Measurement error and biological stochasticity at any one dose contribute less heavily to the final response value, thereby increasing the signal-to-noise ratio and making AUC a more suitable target for machine learning models.

The results confirm this expectation clearly. Indeed, the comparison between the LFC and AUC benchmarks is as informative as either benchmark considered independently. Since the benchmark structure, preprocessing pipeline, and evaluation methodology are identical between the two notebooks, any differences in performance can be interpreted as genuine differences in the predictability of the two response readouts rather than as methodological artefacts.

# Overall Performance: The Magnitude of the LFC-to-AUC Improvement

The merged results file for the AUC benchmark contains 90,000 rows, corresponding to the same factorial design as the LFC experiment. Across this complete space, the global mean Spearman correlation is 0.117 and the global median is 0.111, representing an approximate fivefold improvement over the LFC global mean of 0.024. The mean R² improves from -1.06 to -0.76, while the median R² improves from -0.133 to -0.083. Although negative R² values remain common, indicating that the null model of predicting the training-set mean is still not universally outperformed, the marked improvement in median R² suggests that a larger fraction of drug-model combinations now exceed this baseline.

This fivefold increase is not simply a rescaling of the same underlying signal. Rather, it reflects a genuine increase in the amount of biologically informative structure that the model can extract from genomic and proteomic features. Under the single-dose LFC setting, the observed response depends strongly on where the chosen dose lies relative to the inflection point of the dose-response curve for a given cell line. Two cell lines may therefore have meaningfully different sensitivity profiles while still exhibiting similar viability at 2.5 µM. AUC reduces this ambiguity by integrating across the full dose range, so that cell lines with genuinely different sensitivities generate genuinely different targets for the model to learn.

Seed variance remains negligible in the AUC experiment. The best-performing seed, 19537, achieves a mean Spearman of 0.1175, whereas the worst-performing seed, 1584678, achieves 0.1167. The difference of 0.0008 is scientifically trivial. As in the LFC benchmark, this stability indicates that the `GroupKFold` cross-validation strategy is producing reliable and reproducible cohort assignments and that the results are not materially affected by seed-specific random initialisation.

# Best and Worst Feature Combinations

The best-performing configuration across all three seeds is `prot_combined_union` with ElasticNet using RNA features alone, achieving a mean Spearman of approximately 0.197 to 0.199 and a median above 0.209. This result is notable for two reasons. First, the winning model is linear. In contrast to the LFC benchmark, where tree ensembles such as Extra Trees were dominant, the smoother and more stable AUC response appears sufficiently well-conditioned for ElasticNet's linear inductive bias to perform strongly. Second, the winning feature set is again RNA alone rather than a multi-modal combination, which may initially seem to conflict with the expectation that proteomics should add value. This result should, however, be interpreted cautiously. It reflects the single best-performing arm-model-feature-set combination averaged across all one hundred drugs, and the combined union arm provides 679 cell lines of coverage, enough for ElasticNet to recover a strong linear signal from transcriptomic data alone. As the proteomics uplift analysis later demonstrates, proteomics does contribute meaningful value, but that value is distributed across different drugs and model families rather than being concentrated within the single best global configuration.

The worst-performing combinations differ noticeably from those observed under LFC. Mutation-only features remain the weakest category, but the failures are more severe in the AUC setting. In particular, the RPPA arm with Ridge regression on mutation features achieves a mean Spearman of -0.050 and a mean R² of -9.86, indicating extreme overfitting to noise within the 144-feature RPPA space when conditioned on sparse binary mutation matrices. The combination of a low-dimensional proteomics space and a linear model sensitive to prediction scale appears to create a particularly unstable regime. Under these conditions, Ridge regression produces predictions that are not simply weak, but systematically misaligned with the true response. The identical values for seeds 1584678 and 17052356 in the worst-combination table further suggest that this failure mode is effectively deterministic given the data, rather than being driven by seed-specific randomness.

# Top Predictable Drugs

The highest-performing drugs in the AUC benchmark reflect a different biological landscape from the LFC top list, and this distinction is itself informative. The most predictable drug is SEPANTRONIUM BROMIDE, a survivin inhibitor, with a mean best-combination Spearman of 0.494 and a standard deviation across seeds of only 0.001. This is the clearest example of near-perfect seed stability in the AUC benchmark. Its preferred feature set, `rna+mut` on the ProCan arm under ElasticNet, is biologically interpretable. Survivin, encoded by *BIRC5*, belongs to the inhibitor of apoptosis protein family and exhibits strong cell-type-dependent expression patterns that are captured effectively at both transcript and protein levels. Cell lines with elevated survivin expression are inherently more resistant to survivin inhibition, creating a strong and learnable relationship between molecular state and drug response.

The remainder of the top fifteen drugs is dominated by anti-mitotic and cytotoxic agents, including VELBAN (vinblastine, a tubulin depolymerisation agent), VERUBULIN (a tubulin-binding agent), VINFLUNINE (a vinca alkaloid), CYT-997 (a tubulin inhibitor), COMBRETASTATIN-A-4 (a tubulin polymerisation inhibitor), TOPOTECAN (a topoisomerase I inhibitor), and BRUCEANTIN (a protein synthesis inhibitor). The prominence of tubulin-targeting agents is not accidental. Sensitivity to such drugs is closely linked to proliferation rate, which is itself encoded in the cellular transcriptional state. Rapidly dividing cell lines with high expression of cell-cycle genes are inherently more vulnerable to agents that disrupt mitosis. RNA features capture this proliferative programme effectively, while proteomics can provide additional value by measuring tubulin isoform abundance and post-translational determinants more directly.

BREFELDIN-A is a particularly interesting outlier within the top list. As a secretory pathway disruptor that inhibits ADP-ribosylation factor 1 (ARF1) and blocks ER-to-Golgi transport, its sensitivity is less obviously linked to proliferation. The fact that it performs best with `mut+prot` on `prot_ms_ccle_gygi` under Ridge regression suggests that protein-level information about secretory pathway capacity, rather than broad transcriptional state, is the main determinant of sensitivity. The interaction with mutation status also appears to be captured adequately by a linear model. For this reason, BREFELDIN-A is a strong candidate for deeper interpretability analysis, particularly because the proteomics contribution is mechanistically less obvious than in the anti-mitotic cases.

At the lower end of the spectrum, BIX-01294, a G9a/GLP histone methyltransferase inhibitor, is the least predictable drug, with a best-combination Spearman of only 0.198 and a worst-observed row of -0.318. All of its worst rows are associated with CNV-only or CNV-plus-mutation feature sets on the ProCan and combined union arms, and this pattern persists across all three seeds. This indicates that CNV features are not merely weak for this drug, but actively misleading. A plausible explanation is that G9a inhibitor sensitivity depends on epigenetic context and chromatin accessibility, neither of which is adequately represented by copy number variation. Instead, CNV features may correlate with superficially predictive patterns in the training fold that fail and reverse in the test fold. WP1066, a JAK2 inhibitor, also appears near the bottom of the ranking, consistent with the broader difficulty of predicting JAK-STAT inhibitor sensitivity from static omics measurements when relevant microenvironmental factors are absent.

# Seed Stability and the Biological Meaning of Consistency

An important structural difference between the LFC and AUC top-drug results is visible in the seed-level variability. In the LFC benchmark, most stable drug candidates showed exactly zero standard deviation across seeds, which suggested a likely artefact of deterministic fold-mean computation rather than genuine seed invariance. In the AUC benchmark, by contrast, the top drugs show very small but non-zero standard deviations, such as 0.001 for SEPANTRONIUM BROMIDE and 0.008 for both VELBAN and CR8-(R). This pattern is more consistent with genuine biological reproducibility, in which the predictive signal is strong enough to dominate seed-level variance without eliminating it completely.

TOPOTECAN presents the clearest near-zero standard deviation case in the AUC top list, with a standard deviation of 3.9 × 10⁻⁷. Given the higher signal level associated with AUC, this is more plausibly a real stability phenomenon than the zero-coefficient-of-variation cases observed under LFC. Topotecan is a clinically established topoisomerase I inhibitor with well-characterised sensitivity determinants, particularly *TOP1* expression and replication fork activity, both of which are captured consistently at the transcript and protein levels. A drug whose sensitivity is governed so strongly by a small number of stable molecular features that fold-mean performance becomes effectively seed-invariant is precisely the kind of case for which interpretability analysis is likely to yield clean and biologically meaningful results.

# Worst Predictable Drugs and What They Reveal

The least predictable drugs in the AUC benchmark are consistently associated with very large negative R² values, indicating severe overfitting rather than a simple absence of signal. LDN-212854 achieves a mean best R² of -9.38, while VANOXERINE achieves -4.29 despite having a positive mean Spearman. This combination of positive Spearman and strongly negative R² is entirely possible because the two metrics capture different properties of predictive performance. Spearman measures rank correlation, whereas R² measures variance explained on the original response scale. A model may therefore rank cell lines correctly while producing predicted values that are badly miscalibrated in magnitude.

For Ridge regression in particular, this pattern is likely to arise when regularisation shrinks coefficients strongly towards zero, producing predictions over an artificially narrow range that does not match the true response variance. The negative R² then reflects scale mismatch rather than incorrect ordering. Several of the lowest-performing drugs are concentrated on the RPPA arm with Ridge regression, which is consistent with the earlier observation that the 144-feature RPPA space, combined with Ridge's sensitivity to prediction scale, creates a systematic failure mode for AUC that was less evident under the noisier LFC setting. Once the response variable becomes more informative, as in the AUC benchmark, prediction-scale mismatch becomes more damaging for R² while leaving rank correlation partially intact.

# Comparison with LFC: What Changes and What Remains Consistent

Comparing the LFC and AUC benchmarks side by side reveals both the task-specific character of some findings and the robustness of others. Absolute performance improves by approximately fivefold across the main metrics, confirming that AUC is the more tractable prediction target and should serve as the primary readout for downstream modelling stages. The drug-level ranking also changes substantially. Only a small number of drugs, such as IACS-10759, appear in both top lists, and the biological profile of the top-ranked drugs shifts from a mixed set of targeted agents and cytotoxics in the LFC benchmark to a much clearer dominance of anti-mitotic and cytotoxic compounds in the AUC benchmark. This likely reflects the fact that AUC is more sensitive to the proliferative-state determinants exploited by anti-mitotic drugs, whereas single-dose LFC is more strongly influenced by the specific pharmacological context at 2.5 µM.

At the same time, several structural findings remain unchanged. Seed stability remains high in both benchmarks. Mutation-only feature sets remain the clearest failure mode. No single proteomics arm or model family emerges as universally optimal. The transition from LFC to AUC therefore sharpens the interpretation of the original LFC findings rather than contradicting them. Taken together, the two benchmarks provide a more complete account of what can and cannot be predicted from static multi-modal omics data within the PRISM setting.

# Summary

The AUC benchmark demonstrates that drug sensitivity prediction from multi-modal genomic and proteomic data becomes substantially more feasible when a dose-aggregated response readout is used. The global mean Spearman of 0.117 represents an approximate fivefold improvement over the LFC baseline, and the best-performing drugs reach Spearman correlations approaching 0.5, which is both practically meaningful and biologically interpretable. The dominance of anti-mitotic agents in the top-drug list reflects the well-established relationship between proliferative state and cytotoxic drug sensitivity, while the emergence of proteomics-favoured feature sets for mechanistically distinct drugs such as BREFELDIN-A and VERUBULIN points to a more drug-class-specific utility for protein-level measurements.

The catastrophic performance of RPPA-based mutation feature sets exposes the limitations of RPPA for AUC prediction more clearly than was possible under the noisier LFC benchmark, thereby strengthening the case for deprioritising RPPA in subsequent modelling stages. Taken together with the accompanying EDA extension results, these findings provide the full empirical basis for the proteomics backbone lock decision.


In [8]:
# Per-drug uplift distribution 

def compute_per_drug_uplift(drug_means_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (seed, arm, model, compound_id): delta Spearman when adding PROT
    to the rna+cnv+mut backbone. Only rows where both feature sets exist are kept.
    """
    base = drug_means_df[drug_means_df["feature_set"] == "rna+cnv+mut"][
        ["seed", "arm", "model", "compound_id", "spearman", "r2"]
    ].rename(columns={"spearman": "sp_base", "r2": "r2_base"})

    full = drug_means_df[drug_means_df["feature_set"] == "rna+cnv+mut+prot"][
        ["seed", "arm", "model", "compound_id", "spearman", "r2"]
    ].rename(columns={"spearman": "sp_full", "r2": "r2_full"})

    joined = full.merge(base, on=["seed", "arm", "model", "compound_id"], how="inner")
    joined["delta_spearman"] = joined["sp_full"] - joined["sp_base"]
    joined["delta_r2"] = joined["r2_full"] - joined["r2_base"]
    # A delta > 0.01 is treated as a meaningful improvement (not noise)
    joined["prot_helped"] = joined["delta_spearman"] > 0.01
    return joined

# Load the merged per-drug aggregated file produced above
per_drug_uplift = compute_per_drug_uplift(merged)

uplift_dist_path = OUT_REPORTS / f"per_drug_uplift_distribution_{TARGET}.csv"
per_drug_uplift.to_csv(uplift_dist_path, index=False)
print("Wrote:", uplift_dist_path, per_drug_uplift.shape)

# Summary: per (arm, model) — fraction of drugs helped, quartiles of delta
uplift_summary = (
    per_drug_uplift
    .groupby(["arm", "model"], as_index=False)
    .agg(
        n_drugs=("compound_id", "nunique"),
        frac_helped=("prot_helped", "mean"),
        mean_delta_sp=("delta_spearman", "mean"),
        median_delta_sp=("delta_spearman", "median"),
        p25_delta=("delta_spearman", lambda x: float(x.quantile(0.25))),
        p75_delta=("delta_spearman", lambda x: float(x.quantile(0.75))),
        mean_sp_base=("sp_base", "mean"),
        mean_sp_full=("sp_full", "mean"),
    )
    .sort_values("mean_delta_sp", ascending=False)
)

uplift_summary_path = OUT_REPORTS / f"uplift_summary_by_arm_model_{TARGET}.csv"
uplift_summary.to_csv(uplift_summary_path, index=False)
print("Wrote:", uplift_summary_path)

print("\nProteomics uplift summary (frac_helped = fraction of drugs where PROT delta > 0.01):")
display(uplift_summary)

# Modality marginal contributions (Shapley-style)

def modality_marginal_contributions(
    drug_means_df: pd.DataFrame,
    model_name: str = "ridge",
) -> pd.DataFrame:
    """
    For each modality M, average Spearman gain from adding M to every subset
    that does not already contain M. Averaged across (drug, arm, seed).
    This is the Shapley value of each modality over the 4-modality game.
    """
    MODS = ["rna", "cnv", "mut", "prot"]
    df = drug_means_df[drug_means_df["model"] == model_name].copy()
    df["fs_set"] = df["feature_set"].apply(lambda s: frozenset(s.split("+")))

    rows = []
    for mod in MODS:
        other_mods = [m for m in MODS if m != mod]
        deltas = []

        for r in range(0, len(MODS)):  # subset sizes 0..3 (subsets not containing mod)
            for without_tuple in _combinations(other_mods, r):
                without_set = frozenset(without_tuple)
                with_set = without_set | {mod}

                sub_with = df[df["fs_set"] == with_set][
                    ["seed", "arm", "compound_id", "spearman"]
                ].rename(columns={"spearman": "sp_with"})

                if len(without_tuple) > 0:
                    sub_without = df[df["fs_set"] == without_set][
                        ["seed", "arm", "compound_id", "spearman"]
                    ].rename(columns={"spearman": "sp_without"})

                    joined = sub_with.merge(
                        sub_without,
                        on=["seed", "arm", "compound_id"],
                        how="inner",
                    )
                    if joined.shape[0] > 0:
                        vals = (joined["sp_with"] - joined["sp_without"]).to_numpy(dtype=float)
                        vals = vals[np.isfinite(vals)]
                        deltas.extend(vals.tolist())
                else:
                    # Adding mod to the empty set: raw Spearman of mod-only
                    if sub_with.shape[0] > 0:
                        vals = sub_with["sp_with"].to_numpy(dtype=float)
                        vals = vals[np.isfinite(vals)]
                        deltas.extend(vals.tolist())

        arr = np.asarray(deltas, dtype=float)
        arr = arr[np.isfinite(arr)]

        rows.append({
            "modality": mod,
            "model": model_name,
            "mean_marginal_contribution": float(np.nanmean(arr)) if arr.size else np.nan,
            "median_marginal_contribution": float(np.nanmedian(arr)) if arr.size else np.nan,
            "std_marginal_contribution": float(np.nanstd(arr)) if arr.size else np.nan,
            "n_comparisons": int(arr.size),
        })

    return pd.DataFrame(rows).sort_values("mean_marginal_contribution", ascending=False)

shapley_rows = []
for m_name in merged["model"].unique():
    shapley_rows.append(modality_marginal_contributions(merged, model_name=m_name))

shapley_df = pd.concat(shapley_rows, ignore_index=True).sort_values(
    ["model", "mean_marginal_contribution"], ascending=[True, False]
)

shapley_path = OUT_REPORTS / f"modality_marginal_contributions_{TARGET}.csv"
shapley_df.to_csv(shapley_path, index=False)
print("Wrote:", shapley_path)

print("\nModality marginal contributions (Shapley-style, per model):")
display(shapley_df)

# Coverage vs performance join

def coverage_vs_performance(
    coverage_df: pd.DataFrame,
    uplift_summary_df: pd.DataFrame,
    model_name: str = "ridge",
) -> pd.DataFrame:
    """
    Joins arm-level coverage stats with performance uplift.
    Directly supports the backbone lock decision.
    """
    cov = coverage_df[[
        "arm", "n_overlap_with_prism_auc_cells", "overall_missing_pct",
        "n_features", "n_cells_with_any_prot",
    ]].copy()

    perf = uplift_summary_df[uplift_summary_df["model"] == model_name][[
        "arm", "frac_helped", "mean_delta_sp", "median_delta_sp",
        "mean_sp_base", "mean_sp_full",
    ]].copy()

    joined = cov.merge(perf, on="arm", how="left")

    # Composite lock score: same formula as the existing heuristic but now explicit
    joined["lock_score"] = (
        joined["mean_delta_sp"].fillna(-1e9)
        + 0.0002 * joined["n_overlap_with_prism_auc_cells"].astype(float)
        - 0.001  * joined["overall_missing_pct"].fillna(100.0)
    )
    return joined.sort_values("lock_score", ascending=False).reset_index(drop=True)

cov_perf = coverage_vs_performance(prot_backbone_coverage, uplift_summary)
cov_perf_path = OUT_REPORTS / f"prot_arm_coverage_vs_performance_{TARGET}.csv"
cov_perf.to_csv(cov_perf_path, index=False)
print("Wrote:", cov_perf_path)

print("\nCoverage vs performance (backbone lock evidence):")
display(cov_perf)

# Drug-level stability across seeds

def drug_stability_report(
    merged_df: pd.DataFrame,
    model_name: str = "ridge",
    feature_set: str = "rna+cnv+mut+prot",
) -> pd.DataFrame:
    """
    Per (arm, compound_id): mean/std Spearman across seeds.
    Stable + positive drugs are prime candidates for interpretability case studies.
    """
    sub = merged_df[
        (merged_df["model"] == model_name) &
        (merged_df["feature_set"] == feature_set)
    ].copy()

    if sub.shape[0] == 0:
        print(f"[WARN] drug_stability_report: no rows for model={model_name}, feature_set={feature_set}")
        return pd.DataFrame()

    stability = (
        sub.groupby(["arm", "compound_id"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            mean_spearman=("spearman", "mean"),
            std_spearman=("spearman", "std"),
            min_spearman=("spearman", "min"),
            max_spearman=("spearman", "max"),
            mean_r2=("r2", "mean"),
        )
    )

    # CV = std / |mean|; low CV means consistent across seeds
    stability["cv_spearman"] = stability["std_spearman"] / (stability["mean_spearman"].abs() + 1e-9)

    # Flag drugs that are positive AND stable across ALL seeds seen
    stability["is_stable_positive"] = (
        (stability["min_spearman"] > 0.05) &   # positive in every seed
        (stability["cv_spearman"] < 0.30)       # low variance across seeds
    )

    return stability.sort_values(
        ["is_stable_positive", "mean_spearman"],
        ascending=[False, False],
    ).reset_index(drop=True)

stability_df = drug_stability_report(merged)

stability_path = OUT_REPORTS / f"drug_stability_across_seeds_{TARGET}.csv"
stability_df.to_csv(stability_path, index=False)
print("Wrote:", stability_path)

stable_candidates = stability_df[stability_df["is_stable_positive"]].head(20)
print(f"\nStable + positive drugs (candidates for Step 14 case studies): {stable_candidates.shape[0]} found")
display(stable_candidates)

Wrote: artifacts/reports/notebook 3b/per_drug_uplift_distribution_auc.csv (6000, 11)
Wrote: artifacts/reports/notebook 3b/uplift_summary_by_arm_model_auc.csv

Proteomics uplift summary (frac_helped = fraction of drugs where PROT delta > 0.01):


,arm,model,n_drugs,frac_helped,mean_delta_sp,median_delta_sp,p25_delta,p75_delta,mean_sp_base,mean_sp_full
13,prot_procan_depmapSanger,ridge,100,0.756667,0.032566,0.028592,0.010861,0.055917,0.126738,0.159305
12,prot_procan_depmapSanger,randomforest,100,0.596667,0.028717,0.023713,-0.010760,0.060366,0.114680,0.143397
14,prot_procan_depmapSanger,xgb_quick,100,0.606667,0.025992,0.030291,-0.015825,0.064325,0.096732,0.122723
3,prot_combined_union,ridge,100,0.610000,0.024506,0.020487,-0.012217,0.059588,0.110204,0.134710
11,prot_procan_depmapSanger,extratrees,100,0.576667,0.023200,0.019423,-0.018681,0.052104,0.147110,0.170311
8,prot_ms_ccle_gygi,ridge,100,0.570000,0.019209,0.019307,-0.006029,0.046652,0.157908,0.177117
7,prot_ms_ccle_gygi,randomforest,100,0.573333,0.018683,0.019957,-0.011870,0.045743,0.125254,0.143937
6,prot_ms_ccle_gygi,extratrees,100,0.530000,0.017903,0.015120,-0.019214,0.048173,0.138051,0.155954
9,prot_ms_ccle_gygi,xgb_quick,100,0.533333,0.017413,0.014813,-0.030985,0.063563,0.105732,0.123145
10,prot_procan_depmapSanger,elasticnet,100,0.516667,0.014038,0.012760,-0.016732,0.037739,0.151302,0.165340


Wrote: artifacts/reports/notebook 3b/modality_marginal_contributions_auc.csv

Modality marginal contributions (Shapley-style, per model):


,modality,model,mean_marginal_contribution,median_marginal_contribution,std_marginal_contribution,n_comparisons
0,rna,elasticnet,0.071113,0.052518,0.104340,9598
1,prot,elasticnet,0.045315,0.012934,0.092850,9598
2,cnv,elasticnet,0.011546,-0.002543,0.097816,9598
3,mut,elasticnet,0.003507,0.002526,0.047324,9598
4,rna,extratrees,0.076318,0.061525,0.096328,9600
5,prot,extratrees,0.040873,0.023206,0.088506,9600
6,cnv,extratrees,0.028394,0.015513,0.083830,9600
7,mut,extratrees,-0.006925,-0.007090,0.045122,9600
8,rna,randomforest,0.069314,0.056531,0.086317,9600
9,prot,randomforest,0.038922,0.024675,0.079951,9600


Wrote: artifacts/reports/notebook 3b/prot_arm_coverage_vs_performance_auc.csv

Coverage vs performance (backbone lock evidence):


,arm,n_overlap_with_prism_auc_cells,overall_missing_pct,n_features,n_cells_with_any_prot,frac_helped,mean_delta_sp,median_delta_sp,mean_sp_base,mean_sp_full,lock_score
0,prot_rppa_ccle,366,0.000000,144,612,0.230000,0.003473,0.002482,0.109205,0.112678,0.076673
1,prot_procan_depmapSanger,273,34.854124,7906,485,0.756667,0.032566,0.028592,0.126738,0.159305,0.052312
2,prot_combined_union,372,58.871158,18751,679,0.610000,0.024506,0.020487,0.110204,0.134710,0.040035
3,prot_ms_ccle_gygi,214,25.025104,11780,304,0.570000,0.019209,0.019307,0.157908,0.177117,0.036984


Wrote: artifacts/reports/notebook 3b/drug_stability_across_seeds_auc.csv

Stable + positive drugs (candidates for Step 14 case studies): 20 found


,arm,compound_id,n_seeds,mean_spearman,std_spearman,min_spearman,max_spearman,mean_r2,cv_spearman,is_stable_positive
0,prot_procan_depmapSanger,ANGUIDINE (BRD:BRD-K45724504-001-01-6),3,0.380203,0.078040,0.290091,0.425259,-0.079326,0.205258,True
1,prot_procan_depmapSanger,SB-939 (BRD:BRD-K86797399-001-05-1),3,0.377710,0.020064,0.366126,0.400878,-0.033980,0.053121,True
2,prot_procan_depmapSanger,BRUCEANTIN (BRD:BRD-A36057565-001-01-0),3,0.376663,0.051911,0.316721,0.406634,-0.014685,0.137819,True
3,prot_ms_ccle_gygi,CYT-997 (BRD:BRD-K23363278-001-02-1),3,0.375866,0.002603,0.372861,0.377368,-0.035214,0.006924,True
4,prot_ms_ccle_gygi,BREFELDIN-A (BRD:BRD-K77841042-001-14-1),3,0.373802,0.018253,0.352726,0.384341,0.009310,0.048830,True
5,prot_procan_depmapSanger,BERZOSERTIB (BRD:BRD-K04701033-001-03-9),3,0.370751,0.020737,0.346805,0.382723,0.032783,0.055933,True
6,prot_combined_union,CR8-(R) (BRD:BRD-K40331046-305-01-5),3,0.360294,0.008381,0.355455,0.369972,-0.029823,0.023262,True
7,prot_ms_ccle_gygi,BRUCEANTIN (BRD:BRD-A36057565-001-01-0),3,0.354059,0.022346,0.328256,0.366960,-0.351636,0.063113,True
8,prot_procan_depmapSanger,BNC105 (BRD:BRD-K20468903-001-01-6),3,0.352749,0.024628,0.324311,0.366968,-0.079691,0.069818,True
9,prot_ms_ccle_gygi,CR8-(R) (BRD:BRD-K40331046-305-01-5),3,0.346666,0.033436,0.327362,0.385275,-0.240469,0.096451,True


# Purpose and Analytical Context

The EDA extensions applied to the AUC benchmark are structurally identical to those used in the LFC experiment: per-drug uplift distribution, Shapley-style modality decomposition, coverage-versus-performance analysis, and drug stability reporting. Because the analytical methodology is held constant between the two notebooks, any differences in the results can be attributed entirely to the change in drug response readout, namely the transition from single-dose log-fold-change to area under the dose-response curve. This design makes the comparison between LFC and AUC one of the most scientifically informative outputs of the benchmark stage, because it reveals which findings are specific to the response formulation and which reflect robust properties of the data and modelling framework.

The central theme that emerges from this comparison is one of amplification. Patterns that were present but relatively weak in the LFC setting become stronger and more clearly interpretable in the AUC setting, while a smaller number of findings change direction altogether. Distinguishing between these categories is essential for making well-founded modelling decisions in the subsequent stages of the project.

# Per-Drug Uplift Distribution

## The Fundamental Shift from LFC to AUC

In the LFC experiment, the strongest proteomics uplift observed across all arm-model combinations was a `frac_helped` value of 0.52 with a mean delta of +0.011 Spearman, achieved by the combined union arm under Ridge regression. This was the only combination for which proteomics benefited a majority of drugs. In the AUC experiment, this picture changes substantially. Ten of the twenty arm-model combinations now achieve a mean delta above +0.009, and the best configuration, `prot_procan_depmapSanger` with Ridge regression, reaches a `frac_helped` value of 0.757 and a mean delta of +0.033. In practical terms, this means that when predicting dose-aggregated drug sensitivity, ProCan proteomics contributes meaningful predictive signal for more than three quarters of all evaluated drugs. This is not a marginal extension of the LFC result, but rather a qualitative shift in the apparent utility of proteomics.

The reason for this change is closely related to the improved predictability of AUC overall. Under the single-dose LFC setting, whether proteomics helps a given drug depends on whether the protein-level signal is sufficiently strong to rise above the noise present in the LFC measurements. In the AUC setting, where the response is less noisy, protein-level information that was previously obscured becomes detectable. As a result, the fraction of drugs for which proteomics contributes positive signal increases substantially.

## Arm-by-Arm Comparison

The ProCan arm exhibits the most dramatic improvement between the LFC and AUC settings. In the LFC benchmark, `prot_procan_depmapSanger` with Ridge regression achieved a `frac_helped` value of 0.38 and a mean delta of +0.005. In the AUC benchmark, the same combination reaches 0.757 and +0.033, corresponding to an approximate sixfold increase in mean delta and a doubling in the proportion of drugs helped. This improvement is not confined to Ridge regression. Random Forest rises from 0.417 and +0.004 in LFC to 0.597 and +0.029 in AUC, while XGBoost improves from 0.460 and +0.007 to 0.607 and +0.026. The magnitude and consistency of this improvement across all five model families strongly suggest that ProCan's deep mass spectrometry proteomics captures biologically meaningful information about dose-response sensitivity that was previously masked by LFC noise.

The `ms_ccle_gygi` arm also improves markedly. Ridge regression moves from a `frac_helped` value of 0.39 and a mean delta of +0.002 in LFC to 0.570 and +0.019 in AUC. This is a substantial gain, though still smaller than that observed for ProCan. This difference is consistent with the smaller PRISM overlap available for `ms_ccle_gygi`, which limits the amount of training signal that can be exploited.

The RPPA arm shows the opposite pattern and constitutes the most important negative result in the AUC EDA. In the LFC experiment, RPPA with Ridge regression achieved a `frac_helped` value of 0.27 and a mean delta of +0.003, weak but still positive. In the AUC experiment, the same combination remains weak at 0.230 and +0.003, while every non-Ridge RPPA model combination now produces either near-zero or negative mean deltas. Most notably, RPPA with ElasticNet achieves a `frac_helped` value of only 0.013, meaning that proteomics helps essentially one or two drugs out of one hundred, with a mean delta that is effectively zero. RPPA with tree ensembles and XGBoost actively reduces performance, with mean deltas ranging from -0.003 to -0.008. This near-complete failure of RPPA under the AUC setting, despite its perfect completeness and largest PRISM overlap, provides strong evidence that RPPA's 144-antibody panel does not capture the protein-level biology most relevant to dose-response sensitivity in this dataset. RPPA was designed primarily to measure signalling activity states through phospho-specific antibodies, which is biologically distinct from the protein abundance information provided by mass spectrometry and appears less useful for the response structure captured by AUC.

## The Relative Decline of the Combined Union Arm

An important reversal occurs for the combined union arm when moving from LFC to AUC. In the LFC experiment, the combined union arm with Ridge regression was the single best-performing configuration in the uplift table, with `frac_helped = 0.52` and mean delta `+0.011`. In the AUC experiment, the same combination ranks fourth overall, with `frac_helped = 0.610` and mean delta `+0.024`, and is clearly surpassed by both ProCan and `ms_ccle_gygi` under Ridge regression and Random Forest. Although the absolute performance of the combined union arm improves, its relative position declines because the other arms, especially ProCan, improve more strongly.

This result is biologically plausible. The combined union arm incorporates all three proteomics platforms, but RPPA now appears to contribute more noise than signal in the AUC setting. As a consequence, the combined feature space becomes diluted relative to the cleaner single-platform spaces provided by ProCan or `ms_ccle_gygi`. In addition, the combined union arm retains substantial structural missingness, with 58.9% overall missingness, meaning that many cell lines contain only partial platform coverage. This creates a more complex modelling problem than that posed by any single proteomics arm. For tree ensembles in particular, the uplift values remain positive but modest, suggesting that the high-dimensional and partially missing feature space is more difficult to exploit effectively than the more coherent feature spaces of ProCan or `ms_ccle_gygi`.

# Modality Marginal Contributions (Shapley-Style Decomposition)

## The ElasticNet NaN Issue

Before interpreting the substantive results, an important technical note is required. The AUC Shapley table originally showed `NaN` values for all ElasticNet modality contributions. This was traced to two `NaN` Spearman values in the RPPA ElasticNet rows, arising from a known failure mode in which strong regularisation causes ElasticNet to predict a constant value for certain drug-fold combinations. Because the modality contribution function originally used `np.mean` rather than `np.nanmean`, these two missing values propagated through the full set of 9,600 comparisons and rendered the entire ElasticNet column invalid. Replacing `np.mean`, `np.median`, and `np.std` with their NaN-safe equivalents resolves the issue. The corrected ElasticNet estimates are therefore based on 9,598 valid comparisons rather than 9,600. Since the missing values constitute less than 0.02% of the total comparison pool, they have negligible influence on the resulting estimates.

## The Increase in the Marginal Value of Proteomics

The comparison between the LFC and AUC Shapley values represents one of the most striking findings of the benchmark. In the LFC experiment, proteomics contributed between +0.005 and +0.007 Spearman across all model families, consistently positive but modest, amounting to roughly one quarter of the contribution of RNA. In the AUC experiment, proteomics contributes between +0.034 and +0.050 Spearman, corresponding to an approximate sixfold increase in absolute magnitude.

The most theoretically important result is observed for Ridge regression. Under AUC, Ridge yields an RNA contribution of +0.058 and a proteomics contribution of +0.050, a ratio of approximately 1.16:1. In other words, for a linear model applied to dose-response prediction, proteomics is nearly as informative as transcriptomics. This would not have been evident from the LFC benchmark alone, where the equivalent Ridge ratio between RNA and proteomics was approximately 2.1:1. The narrowing of this gap under AUC suggests that protein abundance measurements capture post-transcriptional regulation, protein stability, and protein-complex effects that are genuinely relevant to drug response across a dose range, but which were obscured under the noisier single-dose setting.

For tree ensembles such as Extra Trees and Random Forest, proteomics contributes +0.041 and +0.039 respectively in AUC, compared with +0.007 and +0.006 in LFC. The RNA-to-proteomics ratio in these models narrows from roughly 3.5:1 in LFC to approximately 1.9:1 in AUC. This still represents a larger gap than that observed for Ridge regression, which is expected. Tree ensembles are better able to exploit complex interaction structures, and RNA features likely exhibit richer interaction patterns with CNV and mutation data. Proteomics, which is partly correlated with RNA but differs through post-transcriptional biology, may be integrated more efficiently by the simpler additive structure of Ridge regression.

## Modality Ranking Across AUC Models

The modality ranking observed in the LFC benchmark, namely

**RNA >> PROT ≈ CNV >> MUT**

continues to hold only partially in the AUC setting. For tree ensembles and XGBoost, the overall order remains RNA > PROT > CNV > MUT, but the gaps narrow considerably and the contribution of CNV increases substantially. CNV contributes approximately +0.028 to +0.030 in AUC tree models, compared with only +0.002 to +0.004 in LFC. This suggests that structural genomic alterations have a more stable relationship with dose-response sensitivity than with single-dose response. One possible interpretation is that copy number changes in major oncogenes and tumour suppressors create persistent pathway-level alterations that manifest more consistently across a full dose range than at any single concentration.

The most consequential change concerns the mutation modality. In LFC, mutation contributed negatively across all five model families without exception. In AUC, mutation contributes positively for Ridge regression at +0.018, while remaining negative for tree ensembles, between -0.007 and -0.025, and for XGBoost at -0.021. This model-dependent divergence is biologically and statistically interpretable. Ridge regression can detect additive effects of specific oncogenic mutations when those effects are large and sufficiently consistent to survive regularised linear fitting. Tree-based models, by contrast, are more prone to overfitting interaction structures involving sparse mutation features that do not generalise across lineage-aware cross-validation folds. The practical implication is that mutation features may be appropriate for linear AUC models, but should be used cautiously in tree-based approaches.

# Coverage vs Performance: Backbone Lock Evidence

## How the Lock Evidence Changes from LFC to AUC

The coverage-versus-performance table in the AUC benchmark provides a clearer decision signal than its LFC counterpart, chiefly because the performance differences between arms are larger and therefore easier to distinguish. In the LFC setting, mean uplift values ranged from -0.008 to +0.011, a narrow interval in which small differences in coverage could easily dominate the lock score. In the AUC setting, the same range expands from -0.008 to +0.033, making the performance dimension far more informative.

RPPA again ranks first under the composite lock score, with a value of 0.077, because of its zero missingness and overlap with 366 PRISM cell lines. However, the evidence for deprioritising RPPA is even stronger than in the LFC analysis. Its mean uplift delta of +0.003 is less than one tenth of ProCan's +0.033, and its `frac_helped` value of 0.23 means that RPPA improves prediction for fewer than one quarter of drugs. From a modelling perspective, a complete dataset with minimal predictive value can be more problematic than a partially missing dataset with strong predictive value, because it introduces structured noise that appears informative to the model.

ProCan ranks second with a lock score of 0.052, and the gap between RPPA and ProCan is smaller in AUC than in LFC. This narrowing reflects the extent to which ProCan's improved predictive performance lifts its composite score, even though its missingness penalty remains unchanged. If the lock score were recalibrated to assign greater weight to predictive performance, which is arguably justified under the higher-signal AUC setting, ProCan would emerge as the clear first-ranked arm.

## Definitive Backbone Lock Recommendation

Taken together, the LFC and AUC coverage-versus-performance analyses support a clear backbone lock decision. For Track 1 clean ablations, `prot_procan_depmapSanger` should serve as the primary proteomics arm. Its AUC performance is unambiguous, with `frac_helped = 0.757` and mean delta `+0.033` under the best-performing model. Although its 34.9% missingness rate is non-trivial, this missingness is largely driven by structured platform availability rather than random per-protein measurement failure, making it manageable through fold-wise imputation and missingness-aware modelling.

The combined union arm should be retained as the Track 2 missing-modality stress test, where its high structural missingness forms a deliberate part of the experimental design rather than a nuisance factor. The `ms_ccle_gygi` arm should be maintained as a secondary Track 1 arm for cross-platform replication of interpretability findings. RPPA, by contrast, should be formally deprioritised for predictive modelling, although its coverage statistics may still remain useful for characterising the cell line population.

# Drug Stability Across Seeds

## Improvement in the Quality of Stability Evidence

The most important structural difference between the LFC and AUC stability tables lies in the coefficient of variation values. In the LFC benchmark, every entry in the top twenty stable candidates showed a standard deviation of exactly 0.0 and a coefficient of variation of exactly 0.0. This pattern was treated with caution because it likely reflected deterministic fold-mean computation rather than genuine biological seed invariance. In the AUC benchmark, the stable candidates exhibit small but non-zero variation across seeds, with coefficient of variation values ranging from 0.007 for CYT-997 on `ms_ccle_gygi` to 0.205 for ANGUIDINE on ProCan. This constitutes qualitatively stronger evidence of stability. These drugs are stable not because the computation is deterministic, but because predictive performance remains consistently positive despite changes in model initialisation, PCA randomness, and fold composition.

The practical implication for Step 14 interpretability analysis is that the AUC stable candidates represent more trustworthy targets than the LFC stable candidates. A drug such as CYT-997, with `CV = 0.007` and minimum Spearman `= 0.373` across three seeds, offers a much stronger basis for reproducible SHAP or integrated-gradient analysis than a drug with `CV = 0.0` under the LFC setting, where the apparent stability may partly reflect the absence of measurable fold-level variation.

## Cross-Arm Replication as Orthogonal Validation

Several drugs appear in the AUC stable-candidate list for more than one proteomics arm, providing a valuable form of cross-platform validation. ANGUIDINE appears under three arms, ProCan, `ms_ccle_gygi`, and combined union, with mean Spearman values of 0.380, 0.341, and 0.333 respectively. BRUCEANTIN appears for both ProCan and `ms_ccle_gygi`, with values of 0.377 and 0.354. CR8-(R) appears for combined union and `ms_ccle_gygi`, with values of 0.360 and 0.347. PCI-24781 appears for both ProCan and `ms_ccle_gygi`, with values of 0.333 and 0.346.

These cross-arm replications are particularly informative because the two main proteomics arms rely on fundamentally different experimental protocols. ProCan uses tandem mass tag-based quantitative proteomics with deep proteome coverage, whereas `ms_ccle_gygi` is based on label-free mass spectrometry with a distinct cell line panel and sample preparation protocol. A drug that remains consistently predictable on both platforms is therefore unlikely to be benefiting from a platform-specific correlation or batch effect. Instead, the relevant protein features are likely to reflect genuine biological signal.

## Biological Interpretation of the Top Stable Candidates

The AUC stable candidates cluster into biologically meaningful categories that help explain why these drugs are genomically predictable. ANGUIDINE (diacetoxyscirpenol), SB-939 (an HDAC inhibitor), and DACINOSTAT (also an HDAC inhibitor) all target chromatin-regulatory machinery. Sensitivity to HDAC inhibition is known to correlate with baseline histone acetylation state and the expression of specific HDAC isoforms, relationships that are captured at both transcript and protein levels and that remain relatively stable across cell lines. The presence of two HDAC inhibitors in the stable-candidate list is therefore biologically plausible and supports the validity of the proteomics signal for this drug class.

CYT-997 and BERZOSERTIB illustrate two different mechanisms with strong molecular correlates. CYT-997 is a tubulin inhibitor whose sensitivity is strongly related to proliferation rate, as discussed in the main AUC results. BERZOSERTIB is an ATR kinase inhibitor whose sensitivity depends on replication stress and DNA repair capacity, both of which are reflected in the expression of DNA damage response genes and proteins. The fact that BERZOSERTIB shows stable positive performance on the ProCan arm specifically suggests that protein-level measurements of replication-stress biology contribute predictive information beyond transcript abundance alone, which is consistent with the known role of post-translational regulation in the DNA damage response.

NAPABUCASIN, a STAT3 inhibitor, is perhaps the most surprising stable candidate given that WP1066, another JAK-STAT pathway inhibitor, appears near the bottom of the predictability ranking. A plausible distinction lies in mechanism. NAPABUCASIN acts through cancer stem-cell self-renewal pathways involving STAT3 and possibly additional transcriptional programmes, whereas WP1066 is a more selective JAK2/STAT3 inhibitor whose activity may depend more heavily on cytokine signalling context. NAPABUCASIN's broader mechanism may therefore generate more stable genomic correlates across cell lines despite the broader complexity of STAT3-related biology.

# Synthesis: Lessons from the LFC-to-AUC Comparison

Reading the LFC and AUC EDA results together yields three major conclusions that neither benchmark could establish in isolation.

The first is that proteomics is genuinely informative for drug sensitivity prediction, but that this informativeness is substantially masked by the noise inherent in single-dose measurements. The roughly sixfold increase in proteomics' Shapley value between LFC and AUC, together with the increase in `frac_helped` across all arms, provides strong evidence that the protein-level signal is real and biologically meaningful. Under this interpretation, the LFC proteomics contributions should be viewed as lower-bound estimates rather than definitive statements of proteomics utility.

The second conclusion is that RPPA is not a suitable proteomics arm for this prediction task. In the LFC setting, RPPA was simply weakly informative. In the AUC setting, where the improved signal-to-noise ratio makes it easier to separate informative from actively harmful features, RPPA combined with tree ensembles and XGBoost consistently reduces performance. The 144-antibody RPPA panel appears to capture signalling activity states that are not the principal determinants of broad-spectrum dose-response sensitivity across one hundred drugs and multiple cancer lineages.

The third conclusion is that `prot_procan_depmapSanger` is the strongest proteomics arm for AUC prediction by a clear margin, and that the backbone lock decision is therefore supported directly by empirical evidence. ProCan's deep, high-coverage mass spectrometry data captures protein abundance information that is broadly relevant to drug sensitivity, and its superiority across all five model families makes this conclusion robust to model choice. The fact that ProCan's relative advantage over `ms_ccle_gygi` is larger in AUC than in LFC further supports the interpretation that its strength reflects genuine biological signal rather than a simple coverage advantage.

# Summary

The AUC EDA extensions confirm and strengthen the conclusions drawn from the LFC analysis while also introducing several genuinely new findings. Proteomics uplift is substantially larger and more consistent in AUC than in LFC, with ProCan improving more than three quarters of drugs under the strongest model. The Shapley decomposition further shows that proteomics nearly matches RNA in importance for linear models applied to AUC prediction, a result with direct implications for architectural choices in later modelling stages. RPPA is now clearly established as unsuitable for AUC prediction, resolving the ambiguity left by the lock-score heuristic in the LFC analysis. Finally, the drug stability table identifies twenty high-quality candidates for Step 14 interpretability analysis, with genuine non-zero cross-seed variation that makes these stability estimates more credible than their LFC counterparts.

Taken together, these results provide the complete empirical foundation for the backbone lock decision and establish the scientific context for the subsequent modelling stages.


In [9]:
OUT_META_GLOBAL = Path("artifacts") / "metadata"
OUT_META_GLOBAL.mkdir(parents=True, exist_ok=True)

LOCK_PATH = OUT_META_GLOBAL / "proteomics_backbone_lock.json"

# Pull release strings from existing lock.json 
existing_lock_path = Path("artifacts") / "metadata" / "lock.json"
if existing_lock_path.exists():
    with existing_lock_path.open("r") as f:
        existing_lock = json.load(f)
    depmap_release = existing_lock.get("depmap_release", "<fill_in>")
    prism_release  = existing_lock.get("prism_release",  "<fill_in>")
else:
    depmap_release = "<fill_in>"
    prism_release  = "<fill_in>"

backbone_lock = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "depmap_release": depmap_release,
    "prism_release": prism_release,
    "random_seeds": [19537, 1584678, 17052356],
    "n_drugs_benchmarked": 100,

    # Target decisions 
    "primary_target": "auc",
    "secondary_target": "lfc",

    # Arm decisions 
    "track1_primary_arm": "prot_procan_depmapSanger",
    "track1_secondary_arm": "prot_ms_ccle_gygi",
    "track2_stress_test_arm": "prot_combined_union",
    "deprioritised_arm": "prot_rppa_ccle",

    # Evidence summary (AUC, Ridge model, 100 drugs)
    "arm_evidence": {
        "prot_procan_depmapSanger": {
            "auc_frac_helped_ridge": 0.757,
            "auc_mean_delta_spearman_ridge": 0.033,
            "lfc_frac_helped_ridge": 0.380,
            "lfc_mean_delta_spearman_ridge": 0.005,
            "n_overlap_prism_auc_cells": 273,
            "n_overlap_prism_lfc_cells": 395,
            "overall_missing_pct": 34.854,
            "n_features": 7906,
            "auc_lock_score": 0.052,
            "decision": "PRIMARY — Track 1. Highest AUC uplift across all model families. Manageable missingness."
        },
        "prot_ms_ccle_gygi": {
            "auc_frac_helped_ridge": 0.570,
            "auc_mean_delta_spearman_ridge": 0.019,
            "lfc_frac_helped_ridge": 0.390,
            "lfc_mean_delta_spearman_ridge": 0.002,
            "n_overlap_prism_auc_cells": 214,
            "n_overlap_prism_lfc_cells": 277,
            "overall_missing_pct": 25.025,
            "n_features": 11780,
            "auc_lock_score": 0.037,
            "decision": "SECONDARY — Track 1. Used for cross-platform replication of interpretability findings."
        },
        "prot_combined_union": {
            "auc_frac_helped_ridge": 0.610,
            "auc_mean_delta_spearman_ridge": 0.025,
            "lfc_frac_helped_ridge": 0.520,
            "lfc_mean_delta_spearman_ridge": 0.011,
            "n_overlap_prism_auc_cells": 372,
            "n_overlap_prism_lfc_cells": 538,
            "overall_missing_pct": 58.871,
            "n_features": 18751,
            "auc_lock_score": 0.040,
            "decision": "TRACK 2 STRESS-TEST ONLY. High structural missingness exercises modality dropout logic. RPPA noise dilutes signal for Track 1."
        },
        "prot_rppa_ccle": {
            "auc_frac_helped_ridge": 0.230,
            "auc_mean_delta_spearman_ridge": 0.003,
            "auc_frac_helped_elasticnet": 0.013,
            "auc_mean_delta_spearman_elasticnet": 0.0001,
            "lfc_frac_helped_ridge": 0.270,
            "lfc_mean_delta_spearman_ridge": 0.003,
            "n_overlap_prism_auc_cells": 366,
            "n_overlap_prism_lfc_cells": 505,
            "overall_missing_pct": 0.0,
            "n_features": 144,
            "auc_lock_score": 0.077,
            "decision": "DEPRIORITISED. Near-zero AUC uplift. Actively hurts tree ensemble predictions. 144-antibody panel not informative for dose-response sensitivity despite complete coverage."
        }
    },

    # Modality rankings from Shapley analysis 
    "modality_shapley_ranking": {
        "lfc": {
            "ranking": "rna >> prot >= cnv >> mut",
            "mut_direction": "negative across all 5 model families",
            "prot_mean_contribution_range": [0.005, 0.007],
            "rna_mean_contribution_range": [0.012, 0.024]
        },
        "auc": {
            "ranking": "rna > prot >> cnv > mut",
            "mut_direction": "positive for Ridge only, negative for tree ensembles and XGBoost",
            "prot_mean_contribution_range": [0.034, 0.050],
            "rna_mean_contribution_range": [0.058, 0.076],
            "ridge_rna_to_prot_ratio": 1.15,
            "note": "ElasticNet Shapley values excluded due to 2 NaN Spearman values in RPPA arm (fixed with nanmean)"
        }
    },

    # Downstream implications
    "downstream_decisions": {
        "mut_in_models": "Include for linear models (Ridge, ElasticNet) on AUC. Treat with caution for tree ensembles. Exclude from primary feature sets for LFC.",
        "imputation_strategy": "To be determined in Notebook 4 bake-off. Current default: median, fold-wise.",
        "missingness_indicators": "Mandatory for prot_combined_union (Track 2). Decision pending for ProCan and MS pending Notebook 4.",
        "step14_case_study_target": "AUC stable candidates preferred over LFC due to genuine non-zero CV across seeds."
    }
}

with LOCK_PATH.open("w", encoding="utf-8") as f:
    json.dump(backbone_lock, f, indent=2)

print(f"Backbone lock written to: {LOCK_PATH}")
print(f"Primary arm  : {backbone_lock['track1_primary_arm']}")
print(f"Secondary arm: {backbone_lock['track1_secondary_arm']}")
print(f"Track 2 arm  : {backbone_lock['track2_stress_test_arm']}")
print(f"Deprioritised: {backbone_lock['deprioritised_arm']}")
print(f"Locked at    : {backbone_lock['locked_at']}")

Backbone lock written to: artifacts/metadata/proteomics_backbone_lock.json
Primary arm  : prot_procan_depmapSanger
Secondary arm: prot_ms_ccle_gygi
Track 2 arm  : prot_combined_union
Deprioritised: prot_rppa_ccle
Locked at    : 2026-03-27T19:26:29.337913+00:00


In [10]:
# Ranked feature-combination table with explicit proteomics source

def add_protein_source_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["uses_prot"] = out["feature_set"].astype(str).str.contains(r"(?:^|\+)prot(?:\+|$)", regex=True)    
    out["protein_dataset"] = np.where(out["uses_prot"], out["arm"], "none")
    return out

rank_input = add_protein_source_columns(merged)

# Rank exact evaluated combinations: arm + model + feature_set
feature_combo_ranking = (
    rank_input
    .groupby(["arm", "protein_dataset", "uses_prot", "model", "feature_set"], as_index=False)
    .agg(
        n_rows=("compound_id", "size"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        std_spearman=("spearman", "std"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
    )
    .sort_values(
        ["mean_spearman", "median_spearman", "mean_r2", "n_drugs"],
        ascending=[False, False, False, False]
    )
    .reset_index(drop=True)
)

feature_combo_ranking.insert(0, "rank", np.arange(1, len(feature_combo_ranking) + 1))

ranking_path = OUT_REPORTS / f"ranked_feature_combinations_{TARGET}.csv"
feature_combo_ranking.to_csv(ranking_path, index=False)
print("Wrote:", ranking_path)

print("\nTop ranked evaluated combinations:")
display(feature_combo_ranking.head(25))

Wrote: artifacts/reports/notebook 3b/ranked_feature_combinations_auc.csv

Top ranked evaluated combinations:


,rank,arm,protein_dataset,uses_prot,model,feature_set,n_rows,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,median_r2
0,1,prot_combined_union,none,False,elasticnet,rna,300,100,0.197244,0.211174,0.115912,-0.084200,-0.083099
1,2,prot_rppa_ccle,prot_rppa_ccle,True,elasticnet,rna+prot,300,100,0.190045,0.191960,0.110458,-0.089062,-0.091531
2,3,prot_rppa_ccle,none,False,elasticnet,rna,300,100,0.189321,0.191960,0.110307,-0.088649,-0.086369
3,4,prot_combined_union,none,False,elasticnet,rna+mut,300,100,0.189232,0.194448,0.112811,-0.085525,-0.089025
4,5,prot_combined_union,prot_combined_union,True,elasticnet,rna+mut+prot,300,100,0.187864,0.184931,0.104316,-0.099967,-0.103190
5,6,prot_combined_union,prot_combined_union,True,elasticnet,rna+prot,300,100,0.187780,0.197548,0.105719,-0.117952,-0.114033
6,7,prot_procan_depmapSanger,prot_procan_depmapSanger,True,elasticnet,rna+mut+prot,300,100,0.187492,0.200239,0.112450,-0.104414,-0.100230
7,8,prot_rppa_ccle,none,False,elasticnet,rna+mut,300,100,0.187299,0.188876,0.106694,-0.088234,-0.085313
8,9,prot_rppa_ccle,prot_rppa_ccle,True,elasticnet,rna+mut+prot,300,100,0.187174,0.187802,0.106636,-0.088466,-0.084668
9,10,prot_procan_depmapSanger,prot_procan_depmapSanger,True,elasticnet,prot,300,100,0.184262,0.180433,0.122923,-0.039500,-0.043470


# Feature Combination Ranking and Selection of Candidate Configurations for the Imputation Bake-off

## Purpose of the ranking analysis

Following the AUC backbone benchmark, an additional ranking analysis was performed to identify the strongest **feature combination, model, and proteomics-arm configurations** across the full benchmark space. The purpose of this step was not to replace the earlier backbone-lock decision, but to refine the next-stage preprocessing experiment by identifying the most promising **proteomics-containing** combinations for the imputation bake-off.

Two ranking outputs were produced. The first ranked every exact evaluated configuration, defined by the tuple **(arm, model, feature set)**. The second retained only the **best-performing row for each feature set**, regardless of arm or model. Together, these outputs make it possible to distinguish between combinations that are strong in absolute terms and combinations for which proteomics appears in genuinely competitive configurations.


## Interpretation of the ranked combinations

The highest-ranked exact configuration in the entire AUC benchmark was **ElasticNet with RNA alone on the `prot_combined_union` eligible cohort**, with mean Spearman **0.197** and median Spearman **0.211**. This result is important because it confirms that, even under the improved AUC target, **RNA remains the single strongest general-purpose modality** in the benchmark. In other words, the transcriptomic signal continues to dominate the broad average ranking task.

However, several **proteomics-containing configurations** appear immediately below the top RNA-only row. These include:

- `prot_rppa_ccle`, ElasticNet, `rna+prot` with mean Spearman **0.190**
- `prot_combined_union`, ElasticNet, `rna+mut+prot` with mean Spearman **0.188**
- `prot_combined_union`, ElasticNet, `rna+prot` with mean Spearman **0.188**
- `prot_procan_depmapSanger`, ElasticNet, `rna+mut+prot` with mean Spearman **0.187**
- `prot_procan_depmapSanger`, Extra Trees, `rna+prot` with mean Spearman **0.184**

This shows that proteomics can participate in highly competitive AUC configurations, particularly when combined with RNA and, in some cases, mutation features. Proteomics therefore does not emerge as the globally dominant modality, but it is clearly compatible with several of the best-performing multi-modal settings.

At the same time, the ranking must be interpreted with caution. A high absolute ranking does **not necessarily imply strong incremental value from proteomics**. For example, `prot_rppa_ccle`, ElasticNet, `rna+prot` achieved mean Spearman **0.190**, but the closely related RNA-only configuration on the same arm achieved **0.189**. The incremental difference is therefore extremely small. This is consistent with the earlier uplift analysis, which showed that RPPA contributed little additional predictive value despite occasionally appearing in strong absolute configurations. The ranking tables should therefore be read alongside, rather than instead of, the uplift and Shapley analyses.

## Interpretation of the best row per feature set

The `best_row_per_feature_set` table identifies the strongest arm-model realisation of each feature set. Several results are especially informative.

For **RNA-only**, the best-performing version again came from `prot_combined_union` with ElasticNet, confirming the strength of transcriptomic signal on the largest eligible cohort.

For **`rna+prot`**, the best-performing row was `prot_rppa_ccle` with ElasticNet. Despite this, RPPA was not considered an appropriate candidate for downstream imputation experiments because the earlier backbone-lock analysis showed that its proteomics contribution was weak in relative terms and often harmful in uplift-based comparisons.

For **`rna+mut+prot`**, the best-performing row came from `prot_combined_union` with ElasticNet, suggesting that proteomics becomes most competitive when combined with a strong transcriptomic backbone and limited genomic context.

For **`rna+cnv+mut+prot`**, the best-performing row came from `prot_ms_ccle_gygi` with Ridge. This is an important result because it identifies the strongest full-backbone proteomics configuration among the locked arms and provides a natural bridge to the imputation bake-off.


## Principles used to choose combinations for the imputation bake-off

The purpose of the imputation bake-off is to evaluate how missing proteomics values should be handled in the most relevant downstream modelling settings. Therefore, the candidate combinations should satisfy four criteria:

1. They should contain **proteomics**, since the bake-off is specifically about proteomics preprocessing.
2. They should rank highly in the AUC benchmark.
3. They should remain consistent with the **locked arm decisions**, namely:
   - `prot_procan_depmapSanger` as the primary Track 1 arm
   - `prot_ms_ccle_gygi` as the secondary Track 1 arm
   - `prot_combined_union` as the Track 2 stress-test arm
4. They should avoid over-reliance on `prot_rppa_ccle`, which was previously deprioritised despite some strong absolute rows.

For this reason, the top-ranked RPPA combinations were **not** selected for the bake-off. Their absolute ranking was acknowledged, but they were excluded because the earlier AUC uplift results showed that RPPA contributed little meaningful incremental information relative to simpler non-proteomic baselines.


## Top 5 proteomics-containing combinations selected for the imputation bake-off

The following five combinations are the most appropriate candidates for the next-stage imputation bake-off:

| Rank for bake-off | Proteomics arm | Model | Feature set | Mean Spearman | Rationale |
|---|---|---|---|---:|---|
| 1 | `prot_combined_union` | ElasticNet | `rna+mut+prot` | 0.187864 | Best competitive Track 2 proteomics-containing configuration; tests imputation under the hardest structural missingness regime |
| 2 | `prot_combined_union` | ElasticNet | `rna+prot` | 0.187780 | Stronger, simpler Track 2 RNA-proteomics setting; useful for stress-testing missingness handling without CNV |
| 3 | `prot_procan_depmapSanger` | ElasticNet | `rna+mut+prot` | 0.187492 | Highest-ranked proteomics-containing ProCan configuration; natural primary Track 1 candidate |
| 4 | `prot_procan_depmapSanger` | Extra Trees | `rna+prot` | 0.184103 | Strong ProCan RNA-proteomics setting under a non-linear model; useful for testing whether imputation choice is model-sensitive |
| 5 | `prot_ms_ccle_gygi` | Ridge | `rna+cnv+mut+prot` | 0.177117 | Best-performing full-backbone MS-CCLE-Gygi configuration; preserves the secondary Track 1 arm in the bake-off |


## Why these five combinations are appropriate

These five combinations provide a balanced and defensible shortlist.

First, they include the **primary Track 1 arm** (`prot_procan_depmapSanger`) in two strong proteomics-containing forms. This is essential, since ProCan was the strongest arm in the uplift-based backbone decision and should therefore be represented prominently in the next preprocessing experiment.

Second, they include the **secondary Track 1 arm** (`prot_ms_ccle_gygi`) through its strongest full-backbone proteomics configuration. This ensures that the imputation bake-off does not become overly specific to ProCan and retains the possibility of cross-platform replication.

Third, they include the **Track 2 stress-test arm** (`prot_combined_union`) in two highly ranked settings. This is important because combined union represents the most challenging missingness structure and is precisely the scenario in which imputation strategy and missingness-aware design are most likely to matter.

Finally, the selected set covers both **simpler RNA-proteomics settings** and **richer multi-modal settings**. This is methodologically valuable because the optimal imputation strategy may differ depending on whether proteomics is appended to a minimal transcriptomic backbone or integrated into a larger genomic context.

## Overall conclusion

The feature-combination ranking confirms three main points. First, RNA remains the strongest single modality in the AUC benchmark. Second, proteomics participates in several highly competitive multi-modal configurations, especially when combined with RNA. Third, absolute ranking alone is not sufficient to choose downstream candidates; combinations must also be interpreted in light of the earlier uplift-based backbone decision.

Based on this combined evidence, the imputation bake-off should proceed with the following five proteomics-containing candidate configurations:

1. `prot_combined_union` + ElasticNet + `rna+mut+prot`
2. `prot_combined_union` + ElasticNet + `rna+prot`
3. `prot_procan_depmapSanger` + ElasticNet + `rna+mut+prot`
4. `prot_procan_depmapSanger` + Extra Trees + `rna+prot`
5. `prot_ms_ccle_gygi` + Ridge + `rna+cnv+mut+prot`

